In [1]:
# SECTION 1 - INSTALL PACKAGES

# This notebook is the simple full-run version.
# It fetches API data, creates CSV outputs, runs token-based fuzzy matching,
# runs TF-IDF cosine similarity, creates automated validation evidence,
# and uploads the final outputs to SQL Server.

!pip -q install thefuzz python-Levenshtein scikit-learn tqdm sqlalchemy pyodbc

# SQL Server driver for Colab.
# This is needed only if RUN_SQL_UPLOAD = True in Section 2.
!curl -sSL https://packages.microsoft.com/keys/microsoft.asc | gpg --dearmor | sudo tee /usr/share/keyrings/microsoft-prod.gpg > /dev/null
!echo "deb [arch=amd64,arm64,armhf signed-by=/usr/share/keyrings/microsoft-prod.gpg] https://packages.microsoft.com/ubuntu/22.04/prod jammy main" | sudo tee /etc/apt/sources.list.d/mssql-release.list > /dev/null
!sudo apt-get update -qq
!sudo ACCEPT_EULA=Y apt-get install -y -qq msodbcsql18 unixodbc-dev

print("Package and SQL driver installation completed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.3/340.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 41.6 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package odbcinst.
(Reading database ... 118243 files and directories currently installed.)
P

In [2]:
# SECTION 2 - IMPORTS, DRIVE FOLDER, AND SETTINGS

from google.colab import drive
import os
import re
import json
import glob
import time
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

from thefuzz import process, fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

# Google Drive project folder
DRIVE_ROOT = "/content/drive"
BASE_DIR = "/content/drive/MyDrive/LD6053-dissertation"

if not os.path.isdir(os.path.join(DRIVE_ROOT, "MyDrive")):
    try:
        drive.mount(DRIVE_ROOT, force_remount=True, timeout_ms=300000)
    except Exception as e:
        raise RuntimeError("Google Drive did not mount. Check Drive permission and run again.") from e
else:
    print("Google Drive is working.")

RUN_DATE = datetime.now().strftime("%Y%m%d")
RUN_DIR = os.path.join(BASE_DIR, RUN_DATE)
os.makedirs(RUN_DIR, exist_ok=True)

# ==============================================================
# SIMPLE FULL-RUN SETTINGS
# ==============================================================
# CHANGE ONLY THIS VALUE to control the retained data volume.
# The pipeline keeps up to this many newest rows from EACH API.
# It automatically requests more pages until the target is reached,
# the API has no more results, or the 90-day boundary is reached.
ROWS_PER_SOURCE = 300

# This notebook only runs the full pipeline.
RUN_MODE = "DAILY_HISTORY_ONE_VOLUME_CONTROL_RECENT_90_DAYS_SQL_V30"

REFETCH_API_DATA = True
FORCE_REBUILD_MATCHING = True
RUN_SQL_UPLOAD = True
SHOW_PRESENTATION_SUMMARY = True

# Recent public-funding evidence settings.
RECENT_DAYS = 90
API_PAGE_SIZE = 100

# Matching thresholds
FUZZY_THRESHOLD = 90
ML_THRESHOLD_PERCENT = 75.0

# Runtime optimisation for fuzzy matching.
# This keeps fuzzy matching but compares against a TF-IDF shortlist instead of every sponsor name.
# It makes the notebook safer to run before the presentation while keeping the same pipeline stages.
USE_TFIDF_SHORTLIST_FOR_FUZZY = True
FUZZY_CANDIDATE_SHORTLIST = 25

# Automated validation thresholds
AUTO_ACCEPT_FUZZY_THRESHOLD = 90
AUTO_ACCEPT_ML_THRESHOLD_PERCENT = 75.0

# Source controls
RUN_CONTRACTS_FINDER = True
RUN_GTR = True

# SQL upload settings
SQL_SPONSOR_MODE = "MATCHED"  # MATCHED or FULL
SQL_INSERT_CHUNK_SIZE = 1      # one autocommitted row per SQL statement; do not change for large runs
SQL_DELETE_CHUNK_SIZE = 10    # small autocommitted delete statements protect the hosted SQL log
SQL_PROGRESS_EVERY = 500      # progress message frequency only
SQL_RETRY_LIMIT = 3           # retries only for temporary SQL log-full errors
SQL_RETRY_WAIT_SECONDS = 10
SQL_STORAGE_PREFLIGHT = True
SQL_STORAGE_MIN_FREE_MB = 2.0
SQL_FAIL_SAFE_NO_TRACEBACK = True

UPLOAD_BRONZE_SILVER_TO_SQL = False
USE_AUTOMATED_VALIDATION = True
RESET_VALIDATION_TABLE_ON_RUN = True
SQL_TEXT_MAX_LENGTH = 1000
SQL_DROP_RAW_JSON_FOR_SQL = True

# API settings
CONTRACTS_BASE_URL = "https://www.contractsfinder.service.gov.uk/Published/Notices/OCDS/Search"
GTR_BASE_URL = "https://gtr.ukri.org/api/search/project"
GTR_HEADERS = {"Accept": "application/json"}
GTR_ACTIVE_FACET = "c3RhdHVzfEFjdGl2ZXxzdHJpbmc="
GTR_ORG_CACHE_FILE = os.path.join(BASE_DIR, "gtr_organisation_name_cache.csv")

# SQL Server settings
SQL_SERVER = r"sql.bsite.net\MSSQL2016"
SQL_DATABASE = "jorgetomaschabrillon_LD6053dissertation"
SQL_USERNAME = "jorgetomaschabrillon_LD6053dissertation"
SQL_PASSWORD = "Topito1990**"

# Sponsor Register file.
# The newest matching Home Office CSV in the project folder is selected automatically.
# This is useful when a newer Sponsor Register file is added before a presentation run.
def find_latest_sponsor_register_file():
    sponsor_patterns = [
        os.path.join(BASE_DIR, "*Worker_and_Temporary_Worker*.csv"),
        os.path.join(BASE_DIR, "SP_*Worker*.csv")
    ]

    candidates = []
    for pattern in sponsor_patterns:
        candidates.extend(glob.glob(pattern))

    candidates = sorted(set(candidates))
    if not candidates:
        return None

    candidates.sort(key=lambda file_path: os.path.getmtime(file_path), reverse=True)
    return candidates[0]


SPONSOR_FILE = find_latest_sponsor_register_file()


if int(ROWS_PER_SOURCE) <= 0:
    raise ValueError("ROWS_PER_SOURCE must be a positive whole number.")

# All extraction limits are derived from the single user setting above.
# One page per requested row is only a fail-safe maximum; both extractors
# normally stop much earlier as soon as ROWS_PER_SOURCE rows are collected.
SOURCE_ROW_LIMIT = int(ROWS_PER_SOURCE)
CONTRACTS_PAGES_TO_FETCH = SOURCE_ROW_LIMIT
GTR_PAGES_TO_FETCH = SOURCE_ROW_LIMIT
# The Contracts Finder extractor uses one recent-date window, so its
# per-window page maximum is also its total maximum.
CONTRACTS_MONTHS_TO_FETCH = 1
CONTRACTS_PAGES_PER_MONTH = CONTRACTS_PAGES_TO_FETCH

print("Project folder:", BASE_DIR)
print("Run folder:", RUN_DIR)
print("Run mode:", RUN_MODE)
print("Refetch API data:", REFETCH_API_DATA)
print("Force rebuild matching:", FORCE_REBUILD_MATCHING)
print("Run SQL upload:", RUN_SQL_UPLOAD)
print("Show presentation summary:", SHOW_PRESENTATION_SUMMARY)
print("Sponsor file detected:", SPONSOR_FILE)
if SPONSOR_FILE:
    print("Sponsor file modified:", datetime.fromtimestamp(os.path.getmtime(SPONSOR_FILE)).strftime("%Y-%m-%d %H:%M:%S"))
print("Recent evidence window (days):", RECENT_DAYS)
print("API page size:", API_PAGE_SIZE)
print("Requested newest Bronze rows per source:", SOURCE_ROW_LIMIT)
print("API pagination: automatic until the row target or 90-day boundary")


Mounted at /content/drive
Project folder: /content/drive/MyDrive/LD6053-dissertation
Run folder: /content/drive/MyDrive/LD6053-dissertation/20260810
Run mode: DAILY_HISTORY_ONE_VOLUME_CONTROL_RECENT_90_DAYS_SQL_V30
Refetch API data: True
Force rebuild matching: True
Run SQL upload: True
Show presentation summary: True
Sponsor file detected: /content/drive/MyDrive/LD6053-dissertation/!SP_-_Worker_and_Temporary_Worker_Web_Register_-_2026-07-23.csv
Sponsor file modified: 2026-07-23 12:03:54
Recent evidence window (days): 90
API page size: 100
Requested newest Bronze rows per source: 300
API pagination: automatic until the row target or 90-day boundary


In [3]:
# SECTION 3 - GENERAL HELPER FUNCTIONS

def current_timestamp():
    """Return a timestamp suitable for SQL Server datetime columns."""
    return datetime.now(timezone.utc).replace(tzinfo=None)


def runtime_row_count(obj):
    """Return row count for timing logs."""
    try:
        if obj is None:
            return 0
        if isinstance(obj, tuple):
            obj = obj[0]
        if hasattr(obj, "__len__"):
            return int(len(obj))
    except Exception:
        pass
    return None


RUNTIME_LOG = []


def log_runtime_step(step_name, start_time, status="SUCCESS", rows=None, notes=""):
    """Save timing information for dissertation evidence and optimisation."""
    end_time = time.perf_counter()
    duration_seconds = round(end_time - start_time, 2)
    RUNTIME_LOG.append({
        "RunDate": RUN_DATE,
        "RunMode": RUN_MODE,
        "RowsPerSource": ROWS_PER_SOURCE,
        "StepName": step_name,
        "Status": status,
        "Rows": rows,
        "DurationSeconds": duration_seconds,
        "StartedAt": datetime.fromtimestamp(start_time).strftime("%Y-%m-%d %H:%M:%S"),
        "FinishedAt": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Notes": str(notes)[:500]
    })
    print(f"[TIME] {step_name}: {duration_seconds} seconds | {status}")


def run_timed_step(step_name, function_to_run):
    """Run one pipeline step and log how long it takes."""
    step_start = time.perf_counter()
    try:
        result = function_to_run()
        log_runtime_step(step_name, step_start, "SUCCESS", runtime_row_count(result), "")
        return result
    except Exception as e:
        log_runtime_step(step_name, step_start, "FAILED", None, str(e))
        save_runtime_log()
        raise


def save_runtime_log():
    """Save runtime log to the dated Google Drive run folder."""
    if not RUNTIME_LOG:
        return None
    runtime_df = pd.DataFrame(RUNTIME_LOG)
    runtime_path = os.path.join(RUN_DIR, "LD6053_RUNTIME_LOG.csv")
    os.makedirs(os.path.dirname(runtime_path), exist_ok=True)
    runtime_df.to_csv(runtime_path, index=False, encoding="utf-8-sig")
    print("Runtime log saved:", runtime_path)
    return runtime_path


def make_output_path(source, layer, method=None):
    """Build the output filename inside the dated run folder."""
    source = str(source).upper()
    layer = str(layer).upper()

    if method:
        method = str(method).upper().replace("FUZZY", "FUZ")
        filename = f"{source}_{layer}_{method}.csv"
    else:
        filename = f"{source}_{layer}.csv"

    return os.path.join(RUN_DIR, filename)


def clean_organisation_name(value):
    """Standardise organisation names before comparison."""
    if pd.isna(value):
        return ""
    value = str(value).upper().strip()
    value = value.replace("&", " AND ")
    value = re.sub(r"[\.,;:'\"\(\)\[\]\{\}/\\\-]", " ", value)
    value = re.sub(r"\bLTD\b", "LIMITED", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def safe_get(dictionary, path, default=None):
    """Read a nested value safely from a dictionary/list structure."""
    current = dictionary
    for key in path:
        try:
            if isinstance(current, list):
                current = current[key]
            else:
                current = current.get(key, default)
        except Exception:
            return default
        if current is None:
            return default
    return current


def add_metadata_columns(df, layer, source):
    """Add audit columns used in the database and dashboard."""
    df = df.copy()
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    if "record_id" not in df.columns:
        df.insert(0, "record_id", range(1, len(df) + 1))

    df["source_system"] = source
    df["medallion_layer"] = layer
    df["inserted_date"] = now
    df["modified_date"] = now

    if "matching_method" not in df.columns:
        df["matching_method"] = None
    if "matching_accuracy_percent" not in df.columns:
        df["matching_accuracy_percent"] = None

    return df


def save_csv(df, source, layer, method=None):
    """Save a dataframe in the dated Google Drive run folder."""
    output_path = make_output_path(source, layer, method)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df):,} rows -> {output_path}")
    return output_path


def find_existing_csv(source, layer, method=None):
    """Find an existing output CSV in the current run folder, or the newest older run folder."""
    expected_name = os.path.basename(make_output_path(source, layer, method))
    direct_path = os.path.join(RUN_DIR, expected_name)
    if os.path.exists(direct_path):
        return direct_path

    candidates = []
    for folder in glob.glob(os.path.join(BASE_DIR, "*")):
        if os.path.isdir(folder):
            possible = os.path.join(folder, expected_name)
            if os.path.exists(possible):
                candidates.append(possible)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]


def load_existing_csv(source, layer, method=None):
    """Load an existing output CSV if it is available."""
    path = find_existing_csv(source, layer, method)
    if path is None:
        return None, None
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded existing {source}_{layer}{'_' + method if method else ''}: {len(df):,} rows <- {path}")
    return df, path


def get_or_create_csv(source, layer, create_function, method=None, use_existing=True, force_create=False):
    """Use an existing CSV when possible; otherwise create and save a new one."""
    if use_existing and not force_create:
        df, path = load_existing_csv(source, layer, method)
        if df is not None:
            return df, path

    df = create_function()
    path = save_csv(df, source, layer, method)
    return df, path


def normalise_sql_column_names(df):
    """Make column names safe for SQL Server."""
    df = df.copy()
    new_columns = []
    used = set()

    for col in df.columns:
        clean = re.sub(r"[^A-Za-z0-9_]", "_", str(col).strip())
        clean = re.sub(r"_+", "_", clean).strip("_")
        if clean == "":
            clean = "column"
        if re.match(r"^\d", clean):
            clean = "c_" + clean

        base = clean
        number = 2
        while clean.lower() in used:
            clean = f"{base}_{number}"
            number += 1
        used.add(clean.lower())
        new_columns.append(clean)

    df.columns = new_columns
    return df


def prepare_for_sql(df):
    """Convert a dataframe into a safer SQL upload format."""
    df = normalise_sql_column_names(df).copy()

    if SQL_DROP_RAW_JSON_FOR_SQL:
        raw_cols = [c for c in df.columns if c.lower() in ["raw_release_json", "raw_project_json"]]
        if raw_cols:
            df = df.drop(columns=raw_cols)

    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].astype(str).replace({"NaT": None})
        elif df[col].dtype == "object":
            df[col] = df[col].apply(lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, (dict, list)) else x)
            df[col] = df[col].apply(lambda x: x[:SQL_TEXT_MAX_LENGTH] if isinstance(x, str) and len(x) > SQL_TEXT_MAX_LENGTH else x)

    return df.where(pd.notnull(df), None)


def empty_dataframe(columns):
    return pd.DataFrame(columns=list(dict.fromkeys(columns)))


In [4]:
# SECTION 4 - HOME OFFICE SPONSOR REGISTER

def load_sponsor_lookup():
    """Load the Home Office Sponsor Register used as the reference dataset."""
    if SPONSOR_FILE is None:
        raise FileNotFoundError(
            "Sponsor Register CSV not found. Put the Worker_and_Temporary_Worker CSV inside: " + BASE_DIR
        )

    df = pd.read_csv(SPONSOR_FILE, low_memory=False)
    if "Organisation Name" not in df.columns:
        raise ValueError("Sponsor Register must contain a column called Organisation Name.")

    df = df.copy()
    df["sponsor_clean_name"] = df["Organisation Name"].apply(clean_organisation_name)
    df = df[df["sponsor_clean_name"] != ""].copy()

    agg_dict = {}
    for col in df.columns:
        if col == "Route":
            agg_dict[col] = lambda x: " / ".join(sorted(set(str(v) for v in x.dropna().unique())))
        elif col != "sponsor_clean_name":
            agg_dict[col] = "first"

    lookup = df.groupby("sponsor_clean_name", as_index=False).agg(agg_dict)
    lookup.insert(0, "sponsor_id", range(1, len(lookup) + 1))

    rename_map = {col: f"sponsor_{col}" for col in lookup.columns if col not in ["sponsor_id", "sponsor_clean_name"]}
    lookup = lookup.rename(columns=rename_map)
    lookup = add_metadata_columns(lookup, "REFERENCE", "HOME")

    sponsor_names = lookup["sponsor_clean_name"].tolist()
    print(f"Loaded {len(sponsor_names):,} unique sponsor organisations.")
    return lookup, sponsor_names


def attach_sponsor_columns(result_df, sponsor_lookup):
    """Attach sponsor details to matched records."""
    if result_df.empty:
        base_cols = list(result_df.columns)
        sponsor_cols = [c for c in sponsor_lookup.columns if c not in base_cols]
        return empty_dataframe(base_cols + sponsor_cols)

    return result_df.merge(
        sponsor_lookup,
        left_on="matched_sponsor_clean",
        right_on="sponsor_clean_name",
        how="left"
    )


In [5]:
# SECTION 5 - MATCHING FUNCTIONS

def gold_empty_frame(df_silver, sponsor_lookup):
    base_cols = list(df_silver.columns)
    match_cols = [
        "source_organisation_original",
        "source_organisation_clean",
        "matched_sponsor_clean",
        "matching_step",
        "matching_method",
        "matching_accuracy_percent"
    ]
    sponsor_cols = [c for c in sponsor_lookup.columns if c not in base_cols + match_cols]
    return empty_dataframe(base_cols + match_cols + sponsor_cols)


def candidate_empty_frame():
    return empty_dataframe([
        "source_organisation_clean",
        "matched_sponsor_clean",
        "matching_step",
        "matching_method",
        "matching_accuracy_percent"
    ])


def build_tfidf_shortlist(unique_names, sponsor_names, neighbours=25):
    """Return likely sponsor-name candidates for each source name."""
    if not unique_names or not sponsor_names:
        return {}

    neighbour_count = min(int(neighbours), len(sponsor_names))
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    sponsor_matrix = vectorizer.fit_transform(sponsor_names)
    source_matrix = vectorizer.transform(unique_names)

    nn = NearestNeighbors(n_neighbors=neighbour_count, metric="cosine")
    nn.fit(sponsor_matrix)
    distances, indices = nn.kneighbors(source_matrix)

    shortlist = {}
    for row_index, source_clean in enumerate(unique_names):
        shortlist[source_clean] = [
            sponsor_names[int(i)] for i in indices[row_index]
        ]
    return shortlist


def create_fuzzy_candidates(
    df_silver,
    source_org_col,
    sponsor_names,
    source_label
):
    """
    Return one best fuzzy candidate and its score for every unique source
    organisation.

    IMPORTANT:
    The FUZZY_THRESHOLD is NOT applied here. The candidate and score are kept
    for Data Quality evidence even when the score is too low to qualify as a
    fuzzy match.
    """
    if (
        df_silver is None
        or df_silver.empty
        or source_org_col not in df_silver.columns
        or not sponsor_names
    ):
        return candidate_empty_frame()

    source_names = (
        df_silver[source_org_col]
        .apply(clean_organisation_name)
    )
    unique_names = (
        source_names[source_names != ""]
        .drop_duplicates()
        .tolist()
    )

    if not unique_names:
        return candidate_empty_frame()

    sponsor_name_set = set(sponsor_names)

    non_exact_names = [
        name for name in unique_names
        if name not in sponsor_name_set
    ]

    shortlist = {}
    if USE_TFIDF_SHORTLIST_FOR_FUZZY and non_exact_names:
        shortlist = build_tfidf_shortlist(
            non_exact_names,
            sponsor_names,
            neighbours=FUZZY_CANDIDATE_SHORTLIST
        )

    rows = []

    for source_clean in tqdm(
        unique_names,
        desc=f"{source_label} fuzzy candidate scoring"
    ):
        if source_clean in sponsor_name_set:
            matched_clean = source_clean
            score = 100.0
            step = "EXACT"
            method = "EXACT"
        else:
            candidates = shortlist.get(source_clean, sponsor_names)
            match = process.extractOne(
                source_clean,
                candidates,
                scorer=fuzz.token_sort_ratio
            )

            if not match:
                rows.append({
                    "source_organisation_clean": source_clean,
                    "matched_sponsor_clean": None,
                    "matching_step": "TOKEN_SORT_RATIO",
                    "matching_method": "FUZZY",
                    "matching_accuracy_percent": None
                })
                continue

            matched_clean = match[0]
            score = float(match[1])
            step = "TOKEN_SORT_RATIO"
            method = "FUZZY"

        rows.append({
            "source_organisation_clean": source_clean,
            "matched_sponsor_clean": matched_clean,
            "matching_step": step,
            "matching_method": method,
            "matching_accuracy_percent": round(score, 2)
        })

    return pd.DataFrame(rows)


def create_ml_candidates(
    df_silver,
    source_org_col,
    sponsor_names,
    source_label
):
    """
    Return one best TF-IDF/Nearest-Neighbours candidate and its cosine
    similarity score for every unique source organisation.

    IMPORTANT:
    ML_THRESHOLD_PERCENT is NOT applied here. The candidate and score are kept
    for Data Quality evidence even when the score is too low to qualify as an
    ML match.
    """
    if (
        df_silver is None
        or df_silver.empty
        or source_org_col not in df_silver.columns
        or not sponsor_names
    ):
        return candidate_empty_frame()

    source_names = (
        df_silver[source_org_col]
        .apply(clean_organisation_name)
    )
    unique_names = (
        source_names[source_names != ""]
        .drop_duplicates()
        .tolist()
    )

    if not unique_names:
        return candidate_empty_frame()

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 4)
    )
    sponsor_matrix = vectorizer.fit_transform(sponsor_names)
    source_matrix = vectorizer.transform(unique_names)

    nn = NearestNeighbors(
        n_neighbors=1,
        metric="cosine"
    )
    nn.fit(sponsor_matrix)

    distances, indices = nn.kneighbors(source_matrix)

    sponsor_name_set = set(sponsor_names)
    rows = []

    for row_index, source_clean in enumerate(
        tqdm(
            unique_names,
            desc=f"{source_label} TF-IDF candidate scoring"
        )
    ):
        if source_clean in sponsor_name_set:
            matched_clean = source_clean
            score_percent = 100.0
            step = "EXACT"
            method = "EXACT"
        else:
            best_index = int(indices[row_index][0])
            similarity = 1 - float(distances[row_index][0])
            score_percent = max(0.0, min(100.0, similarity * 100))
            matched_clean = sponsor_names[best_index]
            step = "TFIDF_COSINE"
            method = "ML"

        rows.append({
            "source_organisation_clean": source_clean,
            "matched_sponsor_clean": matched_clean,
            "matching_step": step,
            "matching_method": method,
            "matching_accuracy_percent": round(score_percent, 2)
        })

    return pd.DataFrame(rows)


def create_fuzzy_gold(
    df_silver,
    source_org_col,
    sponsor_lookup,
    sponsor_names,
    source_label,
    candidate_df=None
):
    """
    Create the threshold-qualified fuzzy Gold output.

    Candidate scores are calculated separately and preserved for comparison.
    Only exact matches or fuzzy scores >= FUZZY_THRESHOLD enter this Gold file.
    """
    if df_silver.empty or source_org_col not in df_silver.columns:
        return gold_empty_frame(df_silver, sponsor_lookup)

    if candidate_df is None:
        candidate_df = create_fuzzy_candidates(
            df_silver,
            source_org_col,
            sponsor_names,
            source_label
        )

    if candidate_df is None or candidate_df.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    candidates = candidate_df.copy()
    candidates["matching_accuracy_percent"] = pd.to_numeric(
        candidates["matching_accuracy_percent"],
        errors="coerce"
    )

    exact_mask = (
        candidates["matching_method"]
        .astype(str)
        .str.upper()
        .eq("EXACT")
    )

    qualified = candidates[
        exact_mask
        | (
            candidates["matching_accuracy_percent"]
            >= float(FUZZY_THRESHOLD)
        )
    ].copy()

    if qualified.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    df = df_silver.copy()
    df["source_organisation_original"] = df[source_org_col]
    df["source_organisation_clean"] = (
        df[source_org_col]
        .apply(clean_organisation_name)
    )
    df = df[df["source_organisation_clean"] != ""].copy()

    result = df.merge(
        qualified,
        on="source_organisation_clean",
        how="inner"
    )
    result = attach_sponsor_columns(result, sponsor_lookup)
    result = add_metadata_columns(result, "GOLD", source_label)
    return result


def create_ml_gold(
    df_silver,
    source_org_col,
    sponsor_lookup,
    sponsor_names,
    source_label,
    candidate_df=None
):
    """
    Create the threshold-qualified TF-IDF/Nearest-Neighbours Gold output.

    Candidate scores are calculated separately and preserved for comparison.
    Only exact matches or scores >= ML_THRESHOLD_PERCENT enter this Gold file.
    """
    if (
        df_silver.empty
        or source_org_col not in df_silver.columns
        or not sponsor_names
    ):
        return gold_empty_frame(df_silver, sponsor_lookup)

    if candidate_df is None:
        candidate_df = create_ml_candidates(
            df_silver,
            source_org_col,
            sponsor_names,
            source_label
        )

    if candidate_df is None or candidate_df.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    candidates = candidate_df.copy()
    candidates["matching_accuracy_percent"] = pd.to_numeric(
        candidates["matching_accuracy_percent"],
        errors="coerce"
    )

    exact_mask = (
        candidates["matching_method"]
        .astype(str)
        .str.upper()
        .eq("EXACT")
    )

    qualified = candidates[
        exact_mask
        | (
            candidates["matching_accuracy_percent"]
            >= float(ML_THRESHOLD_PERCENT)
        )
    ].copy()

    if qualified.empty:
        return gold_empty_frame(df_silver, sponsor_lookup)

    df = df_silver.copy()
    df["source_organisation_original"] = df[source_org_col]
    df["source_organisation_clean"] = (
        df[source_org_col]
        .apply(clean_organisation_name)
    )
    df = df[df["source_organisation_clean"] != ""].copy()

    result = df.merge(
        qualified,
        on="source_organisation_clean",
        how="inner"
    )
    result = attach_sponsor_columns(result, sponsor_lookup)
    result = add_metadata_columns(result, "GOLD", source_label)
    return result


def has_text(value):
    return pd.notna(value) and str(value).strip() != ""


def score_value(value):
    if pd.isna(value):
        return None
    try:
        return float(value)
    except Exception:
        return None


def automatic_match_decision(
    source_org,
    fuzzy_match,
    ml_match,
    fuzzy_score,
    ml_score
):
    """
    Make the final automated decision while retaining low-score candidate
    evidence.

    Thresholds decide qualification/acceptance here; they do not erase the
    candidate scores before comparison.
    """
    source_text = (
        str(source_org).strip()
        if has_text(source_org)
        else ""
    )
    fuzzy_text = (
        str(fuzzy_match).strip()
        if has_text(fuzzy_match)
        else ""
    )
    ml_text = (
        str(ml_match).strip()
        if has_text(ml_match)
        else ""
    )

    fuzzy_candidate = fuzzy_text != ""
    ml_candidate = ml_text != ""

    fuzzy_score_value = score_value(fuzzy_score)
    ml_score_value = score_value(ml_score)

    fuzzy_ok = (
        fuzzy_candidate
        and fuzzy_score_value is not None
        and fuzzy_score_value >= float(AUTO_ACCEPT_FUZZY_THRESHOLD)
    )
    ml_ok = (
        ml_candidate
        and ml_score_value is not None
        and ml_score_value >= float(AUTO_ACCEPT_ML_THRESHOLD_PERCENT)
    )

    if not fuzzy_candidate and not ml_candidate:
        return (
            None,
            None,
            None,
            "UNMATCHED",
            "No sponsor candidate was produced by either method.",
            0
        )

    exact_match = (
        source_text != ""
        and (
            (fuzzy_candidate and fuzzy_text == source_text)
            or (ml_candidate and ml_text == source_text)
        )
    )

    if exact_match:
        match_text = (
            fuzzy_text if fuzzy_text == source_text
            else ml_text
        )
        return (
            "Exact",
            match_text,
            100.0,
            "ACCEPTED",
            "Exact cleaned-name match to the Sponsor Register.",
            1
        )

    methods_agree = (
        fuzzy_candidate
        and ml_candidate
        and fuzzy_text == ml_text
    )

    available_scores = [
        value
        for value in [fuzzy_score_value, ml_score_value]
        if value is not None
    ]
    best_score = max(available_scores) if available_scores else None

    if methods_agree:
        if fuzzy_ok and ml_ok:
            return (
                "Fuzzy+ML",
                fuzzy_text,
                best_score,
                "ACCEPTED",
                "Fuzzy and TF-IDF selected the same sponsor above both acceptance thresholds.",
                1
            )

        return (
            "Fuzzy+ML",
            fuzzy_text,
            best_score,
            "EXCLUDED_LOW_CONFIDENCE",
            "Both methods selected the same sponsor, but one or both scores were below the acceptance thresholds.",
            0
        )

    if fuzzy_ok and ml_ok:
        if (
            fuzzy_score_value is not None
            and (
                ml_score_value is None
                or fuzzy_score_value >= ml_score_value
            )
        ):
            return (
                "Fuzzy",
                fuzzy_text,
                fuzzy_score_value,
                "EXCLUDED_METHOD_DISAGREEMENT",
                "Fuzzy and TF-IDF both met their thresholds but selected different sponsors.",
                0
            )

        return (
            "ML",
            ml_text,
            ml_score_value,
            "EXCLUDED_METHOD_DISAGREEMENT",
            "Fuzzy and TF-IDF both met their thresholds but selected different sponsors.",
            0
        )

    if fuzzy_ok and not ml_ok:
        return (
            "Fuzzy",
            fuzzy_text,
            fuzzy_score_value,
            "EXCLUDED_SINGLE_METHOD",
            "Only fuzzy matching met its acceptance threshold; the TF-IDF candidate was retained as evidence but did not qualify.",
            0
        )

    if ml_ok and not fuzzy_ok:
        return (
            "ML",
            ml_text,
            ml_score_value,
            "EXCLUDED_SINGLE_METHOD",
            "Only TF-IDF/Nearest-Neighbours met its acceptance threshold; the fuzzy candidate was retained as evidence but did not qualify.",
            0
        )

    # Neither method met its threshold.
    if (
        fuzzy_score_value is not None
        and (
            ml_score_value is None
            or fuzzy_score_value >= ml_score_value
        )
    ):
        return (
            "Fuzzy",
            fuzzy_text if fuzzy_candidate else ml_text,
            best_score,
            "EXCLUDED_LOW_CONFIDENCE",
            "Neither method met its acceptance threshold; both candidate scores were retained for audit evidence.",
            0
        )

    return (
        "ML",
        ml_text if ml_candidate else fuzzy_text,
        best_score,
        "EXCLUDED_LOW_CONFIDENCE",
        "Neither method met its acceptance threshold; both candidate scores were retained for audit evidence.",
        0
    )


def best_unique_matches(df, match_col_name, score_col_name):
    """Keep one candidate row per cleaned source organisation."""
    if (
        df is None
        or df.empty
        or "source_organisation_clean" not in df.columns
    ):
        return empty_dataframe([
            "source_organisation_clean",
            match_col_name,
            score_col_name
        ])

    temp = df[[
        "source_organisation_clean",
        "matched_sponsor_clean",
        "matching_accuracy_percent"
    ]].copy()

    temp = temp.rename(columns={
        "matched_sponsor_clean": match_col_name,
        "matching_accuracy_percent": score_col_name
    })

    temp[score_col_name] = pd.to_numeric(
        temp[score_col_name],
        errors="coerce"
    )

    temp = temp.sort_values(
        by=["source_organisation_clean", score_col_name],
        ascending=[True, False]
    )

    temp = temp.drop_duplicates(
        subset=["source_organisation_clean"],
        keep="first"
    )
    return temp


def create_matching_comparison(
    fuzzy_candidate_df,
    ml_candidate_df,
    source_label
):
    """
    Compare the best fuzzy and ML candidates.

    The comparison keeps candidate names and scores even when they are below
    the thresholds, so excluded rows retain meaningful Data Quality evidence.
    """
    comparison_cols = [
        "source_system",
        "source_organisation_clean",
        "fuzzy_match",
        "fuzzy_score_percent",
        "ml_match",
        "ml_score_percent",
        "methods_agree",
        "fuzzy_found_match",
        "ml_found_match",
        "recommended_method",
        "recommended_match",
        "recommended_score_percent",
        "auto_validation_status",
        "auto_validation_reason",
        "dashboard_eligible"
    ]

    fuzzy_part = best_unique_matches(
        fuzzy_candidate_df,
        "fuzzy_match",
        "fuzzy_score_percent"
    )
    ml_part = best_unique_matches(
        ml_candidate_df,
        "ml_match",
        "ml_score_percent"
    )

    comparison = fuzzy_part.merge(
        ml_part,
        on="source_organisation_clean",
        how="outer"
    )

    if comparison.empty:
        comparison = empty_dataframe(comparison_cols)
    else:
        comparison.insert(0, "source_system", source_label)

        comparison["fuzzy_score_percent"] = pd.to_numeric(
            comparison["fuzzy_score_percent"],
            errors="coerce"
        )
        comparison["ml_score_percent"] = pd.to_numeric(
            comparison["ml_score_percent"],
            errors="coerce"
        )

        comparison["methods_agree"] = (
            comparison["fuzzy_match"]
            .fillna("")
            .astype(str)
            ==
            comparison["ml_match"]
            .fillna("")
            .astype(str)
        ) & (
            comparison["fuzzy_match"].notna()
            & comparison["ml_match"].notna()
        )

        # "Found match" means the method met its own threshold.
        # Candidate names/scores remain in the comparison CSV even when False.
        comparison["fuzzy_found_match"] = (
            comparison["fuzzy_match"].notna()
            & (
                comparison["fuzzy_score_percent"]
                >= float(FUZZY_THRESHOLD)
            )
        )
        comparison["ml_found_match"] = (
            comparison["ml_match"].notna()
            & (
                comparison["ml_score_percent"]
                >= float(ML_THRESHOLD_PERCENT)
            )
        )

        decisions = comparison.apply(
            lambda row: automatic_match_decision(
                row.get("source_organisation_clean"),
                row.get("fuzzy_match"),
                row.get("ml_match"),
                row.get("fuzzy_score_percent"),
                row.get("ml_score_percent")
            ),
            axis=1,
            result_type="expand"
        )

        decisions.columns = [
            "recommended_method",
            "recommended_match",
            "recommended_score_percent",
            "auto_validation_status",
            "auto_validation_reason",
            "dashboard_eligible"
        ]

        comparison = pd.concat(
            [comparison, decisions],
            axis=1
        )

        comparison["methods_agree"] = (
            comparison["methods_agree"].astype(int)
        )
        comparison["fuzzy_found_match"] = (
            comparison["fuzzy_found_match"].astype(int)
        )
        comparison["ml_found_match"] = (
            comparison["ml_found_match"].astype(int)
        )
        comparison["dashboard_eligible"] = (
            comparison["dashboard_eligible"].astype(int)
        )

        comparison = comparison[comparison_cols]

    comparison = add_metadata_columns(
        comparison,
        "EVALUATION",
        source_label
    )
    return comparison


def validate_comparison_score_evidence(df, source_label):
    """
    Fail before SQL upload if comparison evidence loses method scores.

    A produced candidate must have its score. This is the defect that the
    previous notebook allowed for excluded records.
    """
    if df is None or df.empty:
        print(f"{source_label}: no comparison rows to validate.")
        return True

    checks = {
        "fuzzy candidate without fuzzy score": (
            df["fuzzy_match"].notna()
            & pd.to_numeric(
                df["fuzzy_score_percent"],
                errors="coerce"
            ).isna()
        ),
        "ML candidate without ML score": (
            df["ml_match"].notna()
            & pd.to_numeric(
                df["ml_score_percent"],
                errors="coerce"
            ).isna()
        ),
        "recommended match without recommended score": (
            df["recommended_match"].notna()
            & pd.to_numeric(
                df["recommended_score_percent"],
                errors="coerce"
            ).isna()
        )
    }

    failures = {
        name: int(mask.sum())
        for name, mask in checks.items()
        if int(mask.sum()) > 0
    }

    if failures:
        raise RuntimeError(
            f"{source_label} comparison score validation FAILED: "
            + "; ".join(
                f"{name} = {count}"
                for name, count in failures.items()
            )
            + ". SQL upload is stopped so incomplete comparison evidence is not written."
        )

    print(
        f"{source_label}: comparison score validation PASSED "
        f"({len(df):,} rows; candidate scores retained)."
    )
    return True


In [6]:
# SECTION 6 - CONTRACTS FINDER PIPELINE

def fetch_contracts_finder_bronze():
    """Extract Contracts Finder award data using cursor pagination."""
    from urllib.parse import urljoin

    all_rows = []
    seen_evidence_keys = set()

    # Conservative delay to reduce the chance of hitting the public API rate limit.
    CF_REQUEST_DELAY_SECONDS = 3.5
    CF_RATE_LIMIT_WAIT_SECONDS = 300
    CF_RATE_LIMIT_RETRIES = 3

    for month_index in range(CONTRACTS_MONTHS_TO_FETCH):
        start_date = (
            datetime.now(timezone.utc) - timedelta(days=RECENT_DAYS)
        ).strftime("%Y-%m-%d")

        end_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        print(f"Contracts Finder recent publication window: {start_date} to {end_date}")

        next_url = CONTRACTS_BASE_URL
        request_params = {
            "stages": "award",
            "publishedFrom": start_date,
            "publishedTo": end_date,
            "limit": API_PAGE_SIZE
        }

        page_number = 0
        seen_page_fingerprints = set()

        progress = tqdm(
            total=CONTRACTS_PAGES_PER_MONTH,
            desc="Contracts Finder recent award pages"
        )

        try:
            while next_url:
                if (
                    CONTRACTS_PAGES_PER_MONTH is not None
                    and page_number >= CONTRACTS_PAGES_PER_MONTH
                ):
                    break

                rate_limit_attempt = 0

                while True:
                    try:
                        response = requests.get(
                            next_url,
                            params=request_params,
                            timeout=25
                        )
                    except Exception as e:
                        raise RuntimeError(
                            f"Contracts Finder request failed for "
                            f"{start_date} to {end_date}: {e}"
                        ) from e

                    # Official documentation describes HTTP 403 for the
                    # Contracts Finder rate limit, while the live service can
                    # also return HTTP 429. Treat both as the same rate-limit
                    # condition and retry the exact same cursor request.
                    if response.status_code in (403, 429):
                        rate_limit_attempt += 1

                        if rate_limit_attempt > CF_RATE_LIMIT_RETRIES:
                            raise RuntimeError(
                                f"Contracts Finder rate limit persisted after "
                                f"{CF_RATE_LIMIT_RETRIES} retries "
                                f"(last HTTP status {response.status_code}). "
                                "The run is stopping before SQL replacement."
                            )

                        retry_after = response.headers.get("Retry-After")
                        wait_seconds = CF_RATE_LIMIT_WAIT_SECONDS

                        if retry_after:
                            try:
                                wait_seconds = max(
                                    wait_seconds,
                                    int(float(retry_after))
                                )
                            except Exception:
                                pass

                        print(
                            f"\nContracts Finder rate limit returned HTTP "
                            f"{response.status_code}. Waiting {wait_seconds} "
                            f"seconds before retry "
                            f"{rate_limit_attempt}/{CF_RATE_LIMIT_RETRIES}..."
                        )

                        time.sleep(wait_seconds)
                        continue

                    if response.status_code != 200:
                        raise RuntimeError(
                            f"Contracts Finder returned HTTP "
                            f"{response.status_code}. "
                            "The run is stopping before SQL replacement so "
                            "partial Contracts Finder data is not treated as "
                            "a complete snapshot."
                        )

                    break

                payload = response.json()
                releases = payload.get("releases", [])

                # Compatibility fallback if the service wraps the response.
                if not releases:
                    results = payload.get("results", [])
                    releases = [
                        item.get("releases", [None])[0]
                        for item in results
                        if item.get("releases")
                    ]

                if not releases:
                    break

                # Guard against an accidental repeated cursor page.
                page_fingerprint = tuple(
                    sorted(
                        str(rel.get("id") or rel.get("ocid") or "")
                        for rel in releases
                        if rel
                    )
                )

                if page_fingerprint in seen_page_fingerprints:
                    print(
                        f"\nContracts Finder repeated a previously seen page "
                        f"for {start_date} to {end_date}. "
                        "Stopping this date window to avoid duplicate evidence."
                    )
                    break

                seen_page_fingerprints.add(page_fingerprint)

                page_number += 1

                for rel in releases:
                    if not rel:
                        continue

                    tender = rel.get("tender", {}) or {}
                    awards = rel.get("awards", []) or []

                    for award in awards:
                        suppliers = award.get("suppliers", []) or []

                        for supplier in suppliers:
                            supplier_name = supplier.get("name")
                            if not supplier_name:
                                continue

                            evidence_key = (
                                str(rel.get("ocid") or ""),
                                str(rel.get("id") or ""),
                                str(award.get("id") or ""),
                                str(supplier.get("id") or ""),
                                str(supplier_name).strip().upper()
                            )
                            if evidence_key in seen_evidence_keys:
                                continue
                            seen_evidence_keys.add(evidence_key)

                            award_date = award.get("date")
                            if award_date and "T" in str(award_date):
                                award_date = str(award_date).split("T")[0]

                            all_rows.append({
                                "Contract_Title": tender.get("title"),
                                "Contract_Value": safe_get(
                                    award, ["value", "amount"], 0
                                ),
                                "Contract_Currency": safe_get(
                                    award, ["value", "currency"], None
                                ),
                                "Supplier_Name": supplier_name,
                                "Supplier_ID": supplier.get("id"),
                                "Award_Date": award_date,
                                "Award_ID": award.get("id"),
                                "OCID": rel.get("ocid"),
                                "Release_ID": rel.get("id"),
                                "Published_Date": rel.get("date"),
                                "API_Month_Start": start_date,
                                "API_Month_End": end_date,
                                "API_Page": page_number,
                                "raw_release_json": json.dumps(
                                    rel,
                                    ensure_ascii=False
                                )
                            })

                progress.update(1)

                if SOURCE_ROW_LIMIT is not None and len(all_rows) >= SOURCE_ROW_LIMIT:
                    print(f"Contracts Finder collected at least {SOURCE_ROW_LIMIT:,} recent award-supplier rows; stopping API pagination early.")
                    break

                # Follow official cursor pagination through links.next.
                next_link = safe_get(
                    payload,
                    ["links", "next"],
                    None
                )

                if not next_link:
                    break

                next_url = urljoin(response.url, str(next_link))

                # links.next already contains the cursor and search parameters.
                request_params = None

                # Deliberate throttling to reduce rate-limit risk.
                time.sleep(CF_REQUEST_DELAY_SECONDS)

        finally:
            progress.close()

    df = pd.DataFrame(all_rows)

    # Remove duplicate contract/supplier evidence by stable identifiers.
    if not df.empty:
        before_dedup = len(df)

        stable_dedup_columns = [
            "OCID",
            "Release_ID",
            "Award_ID",
            "Supplier_ID",
            "Supplier_Name"
        ]

        stable_dedup_columns = [
            c for c in stable_dedup_columns
            if c in df.columns
        ]

        if stable_dedup_columns:
            df = df.drop_duplicates(
                subset=stable_dedup_columns,
                keep="first"
            )
        else:
            df = df.drop_duplicates()

        removed = before_dedup - len(df)
        print(
            f"Contracts Finder stable-ID deduplication removed "
            f"{removed:,} duplicate row(s)."
        )

        # Hard filter for dashboard relevance: retain only awards dated
        # inside the same recent evidence window, then order newest-first.
        award_dates = pd.to_datetime(df["Award_Date"], errors="coerce", utc=True)
        cutoff_start = pd.Timestamp(start_date, tz="UTC")
        cutoff_end = pd.Timestamp(end_date, tz="UTC") + pd.Timedelta(days=1)
        recent_mask = award_dates.ge(cutoff_start) & award_dates.lt(cutoff_end)
        removed_outside_window = int((~recent_mask).sum())
        df = df.loc[recent_mask].copy()
        df["_AwardDateSort"] = award_dates.loc[recent_mask]
        df = df.sort_values(by="_AwardDateSort", ascending=False).drop(columns=["_AwardDateSort"])
        if SOURCE_ROW_LIMIT is not None and len(df) > SOURCE_ROW_LIMIT:
            print(f"Contracts Finder retained the newest {SOURCE_ROW_LIMIT:,} rows to keep source volumes comparable.")
            df = df.head(SOURCE_ROW_LIMIT).copy()
        print(f"Contracts Finder removed {removed_outside_window:,} row(s) outside the recent award-date window.")
        if not df.empty:
            print(f"Contracts Finder retained {len(df):,} award-supplier row(s) from {df['Award_Date'].min()} to {df['Award_Date'].max()}.")

    df = add_metadata_columns(df, "BRONZE", "CF")
    return df


def clean_contracts_finder_silver(df_bronze):
    """Clean Contracts Finder Bronze data into the Silver layer."""
    if df_bronze.empty:
        columns = [
            "Contract_Title", "Contract_Value", "Contract_Currency", "Supplier_Name", "Supplier_ID",
            "Award_Date", "Award_ID", "OCID", "Release_ID", "Published_Date",
            "API_Month_Start", "API_Month_End", "API_Page", "raw_release_json",
            "Supplier_Name_Clean"
        ]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "CF")

    df = df_bronze.copy()
    df = df.dropna(subset=["Supplier_Name"])
    df["Supplier_Name"] = df["Supplier_Name"].astype(str).str.upper().str.strip()
    df["Supplier_Name_Clean"] = df["Supplier_Name"].apply(clean_organisation_name)
    df["Contract_Value"] = pd.to_numeric(df["Contract_Value"], errors="coerce").fillna(0)
    df["Award_Date"] = pd.to_datetime(df["Award_Date"], errors="coerce").dt.date
    df = df.drop_duplicates()
    df = df.sort_values(by="Award_Date", ascending=False, na_position="last")
    df = add_metadata_columns(df, "SILVER", "CF")
    return df


In [7]:
# SECTION 7 - GATEWAY TO RESEARCH PIPELINE

def fetch_gtr_bronze():
    """Fetch active GTR projects that started within the recent evidence window."""
    all_rows = []
    seen_project_keys = set()
    window_end = datetime.now(timezone.utc).date()
    window_start = (datetime.now(timezone.utc) - timedelta(days=RECENT_DAYS)).date()
    maximum_pages = GTR_PAGES_TO_FETCH if GTR_PAGES_TO_FETCH is not None else 1000000

    def parse_gtr_date(value):
        if value is None:
            return None
        try:
            if isinstance(value, (int, float, np.integer, np.floating)):
                parsed = pd.to_datetime(value, unit="ms", utc=True, errors="coerce")
            else:
                parsed = pd.to_datetime(value, utc=True, errors="coerce")
            return None if pd.isna(parsed) else parsed.date()
        except Exception:
            return None

    print(f"GTR recent start-date window: {window_start} to {window_end}")
    progress = tqdm(total=None if GTR_PAGES_TO_FETCH is None else maximum_pages, desc="GTR newest active project pages")

    try:
        for page in range(1, maximum_pages + 1):
            params = {
                "term": "*",
                "fetchSize": API_PAGE_SIZE,
                "page": page,
                "selectedSortableField": "pro.sd",
                "selectedSortOrder": "DESC",
                "selectedFacets": GTR_ACTIVE_FACET,
                "fields": ""
            }

            response = None
            for attempt in range(1, 4):
                try:
                    response = requests.get(GTR_BASE_URL, headers=GTR_HEADERS, params=params, timeout=40)
                    if response.status_code == 200:
                        break
                    if response.status_code not in (429, 500, 502, 503, 504):
                        raise RuntimeError(f"GTR returned HTTP {response.status_code}")
                except Exception as error:
                    if attempt == 3:
                        raise RuntimeError(f"GTR page {page} failed after 3 attempts: {error}") from error
                if attempt < 3:
                    time.sleep(5 * attempt)

            if response is None or response.status_code != 200:
                status = None if response is None else response.status_code
                raise RuntimeError(f"GTR page {page} failed after 3 attempts (HTTP {status}).")

            search_data = response.json().get("facetedSearchResultBean", {})
            if page == 1:
                if search_data.get("appliedSortField") != "pro.sd" or search_data.get("appliedSortOrder") != "DESC":
                    raise RuntimeError("GTR did not confirm newest-first start-date ordering.")
                print("GTR active projects reported:", f"{int(search_data.get('totalResults', 0)):,}")

            results = search_data.get("results", []) or []
            if not results:
                break

            page_dates = []
            for result in results:
                composition = result.get("projectComposition", {}) or {}
                project = composition.get("project", {}) or {}
                fund_info = project.get("fund", {}) if isinstance(project.get("fund", {}), dict) else {}
                start_date = parse_gtr_date(fund_info.get("start"))
                end_date = parse_gtr_date(fund_info.get("end"))

                if start_date is not None:
                    page_dates.append(start_date)

                # Hard filter: no future projects and nothing older than RECENT_DAYS.
                if start_date is None or start_date < window_start or start_date > window_end:
                    continue

                lead_org = composition.get("leadResearchOrganisation", {}) or {}
                org_url = lead_org.get("resourceUrl") or "Unknown"
                lead_name = lead_org.get("name") or "Unknown"

                project_key = str(
                    project.get("id")
                    or project.get("resourceUrl")
                    or f"{project.get('grantReference')}|{project.get('title')}|{start_date.isoformat()}"
                )
                if project_key in seen_project_keys:
                    continue
                seen_project_keys.add(project_key)

                all_rows.append({
                    "project_id": project.get("id") or project_key,
                    "project_url": project.get("resourceUrl"),
                    "title": project.get("title"),
                    "status": "ACTIVE",
                    "grant_reference": project.get("grantReference"),
                    "grant_category": project.get("grantCategory"),
                    "abstract_text": project.get("abstractText"),
                    "start_date": start_date.isoformat(),
                    "end_date": None if end_date is None else end_date.isoformat(),
                    "fund_type": fund_info.get("type"),
                    "value_pounds": fund_info.get("valuePounds"),
                    "funder_name": safe_get(fund_info, ["funder", "name"], None),
                    "funder_id": safe_get(fund_info, ["funder", "id"], None),
                    "org_url": org_url,
                    "lead_organisation": str(lead_name).upper().strip(),
                    "API_Page": page,
                    "raw_project_json": json.dumps(project, ensure_ascii=False)
                })

            progress.update(1)

            if SOURCE_ROW_LIMIT is not None and len(all_rows) >= SOURCE_ROW_LIMIT:
                print(f"GTR collected at least {SOURCE_ROW_LIMIT:,} recent project rows; stopping API pagination early.")
                break

            # Results are newest-first. Once a page crosses the lower date
            # boundary, later pages cannot contain records in the window.
            if page_dates and min(page_dates) < window_start:
                print(f"GTR reached records older than {window_start} on page {page}; stopping early.")
                break

            time.sleep(0.2)
    finally:
        progress.close()

    df = pd.DataFrame(all_rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["project_id"], keep="first")
        df = df.sort_values(by="start_date", ascending=False)
        if SOURCE_ROW_LIMIT is not None and len(df) > SOURCE_ROW_LIMIT:
            print(f"GTR retained the newest {SOURCE_ROW_LIMIT:,} rows to keep source volumes comparable.")
            df = df.head(SOURCE_ROW_LIMIT).copy()
        print(f"GTR retained {len(df):,} project row(s) from {df['start_date'].min()} to {df['start_date'].max()}.")
    df = add_metadata_columns(df, "BRONZE", "GTR")
    return df


def load_gtr_org_cache():
    if not os.path.exists(GTR_ORG_CACHE_FILE):
        return {}
    try:
        cache_df = pd.read_csv(GTR_ORG_CACHE_FILE)
        return dict(zip(cache_df["org_url"].astype(str), cache_df["lead_organisation"].astype(str)))
    except Exception:
        return {}


def save_gtr_org_cache(cache):
    try:
        cache_df = pd.DataFrame({"org_url": list(cache.keys()), "lead_organisation": list(cache.values())})
        cache_df.to_csv(GTR_ORG_CACHE_FILE, index=False, encoding="utf-8-sig")
    except Exception as e:
        print("Could not save GTR organisation cache:", e)


def fetch_gtr_organisation_name(org_url):
    """Fetch the lead organisation name from a GTR organisation URL."""
    if not org_url or org_url == "Unknown":
        return "Unknown"
    try:
        secure_url = str(org_url).replace("http://", "https://")
        res = requests.get(secure_url, headers=GTR_HEADERS, timeout=10)
        if res.status_code == 200:
            return str(res.json().get("name", "Unknown")).upper().strip()
    except Exception:
        pass
    return "Unknown"


def clean_gtr_silver(df_bronze):
    """Clean GTR Bronze data into the Silver layer and add lead organisation names."""
    if df_bronze.empty:
        columns = [
            "project_id", "project_url", "title", "status", "grant_reference", "grant_category",
            "abstract_text", "start_date", "end_date", "fund_type", "value_pounds",
            "funder_name", "funder_id", "org_url", "API_Page", "raw_project_json",
            "lead_organisation", "lead_organisation_clean"
        ]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    df = df_bronze.copy()
    df = df[df["status"].astype(str).str.upper() == "ACTIVE"].copy()

    if df.empty:
        columns = list(df_bronze.columns) + ["lead_organisation", "lead_organisation_clean"]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    # The newest-first GTR search endpoint already includes the lead
    # organisation. Fall back to the older per-organisation lookup only
    # when an older cached Bronze file lacks that field.
    if "lead_organisation" in df.columns:
        df["lead_organisation"] = df["lead_organisation"].fillna("Unknown").astype(str).str.upper().str.strip()
    else:
        unique_urls = [u for u in df["org_url"].dropna().unique() if u != "Unknown"]
        url_to_name = load_gtr_org_cache()
        url_to_name["Unknown"] = "Unknown"

        missing_urls = [u for u in unique_urls if str(u) not in url_to_name]
        for url in tqdm(missing_urls, desc="Fetching GTR lead organisation names"):
            url_to_name[str(url)] = fetch_gtr_organisation_name(url)
            time.sleep(0.05)

        if missing_urls:
            save_gtr_org_cache(url_to_name)

        df["lead_organisation"] = df["org_url"].astype(str).map(url_to_name).fillna("Unknown")
    df = df[df["lead_organisation"].astype(str).str.upper() != "UNKNOWN"].copy()

    if df.empty:
        columns = list(df_bronze.columns) + ["lead_organisation", "lead_organisation_clean"]
        return add_metadata_columns(empty_dataframe(columns), "SILVER", "GTR")

    df["lead_organisation_clean"] = df["lead_organisation"].apply(clean_organisation_name)
    df["value_pounds"] = pd.to_numeric(df["value_pounds"], errors="coerce")
    df["start_date"] = pd.to_datetime(df["start_date"], errors="coerce").dt.date
    df["end_date"] = pd.to_datetime(df["end_date"], errors="coerce").dt.date
    df = df.drop_duplicates()
    df = df.sort_values(by="start_date", ascending=False, na_position="last")
    df = add_metadata_columns(df, "SILVER", "GTR")
    return df


In [8]:
# SECTION 8 - RUN FULL PIPELINE AND SAVE CSV OUTPUTS

created_files = []
tables_to_upload = {}

# Sponsor lookup. The Sponsor Register is the reference list used for matching.
sponsor_lookup, sponsor_names = run_timed_step(
    "Load Home Office Sponsor Register",
    load_sponsor_lookup
)
created_files.append(save_csv(sponsor_lookup, "HOME", "SPONSOR"))

if RUN_CONTRACTS_FINDER:
    print("\n================ CONTRACTS FINDER PIPELINE ================")

    cf_bronze = run_timed_step(
        "Contracts Finder API extraction - Bronze",
        fetch_contracts_finder_bronze
    )
    created_files.append(save_csv(cf_bronze, "CF", "BRONZE"))
    tables_to_upload["CFBronze"] = cf_bronze

    cf_silver = run_timed_step(
        "Contracts Finder cleaning - Silver",
        lambda: clean_contracts_finder_silver(cf_bronze)
    )
    created_files.append(save_csv(cf_silver, "CF", "SILVER"))
    tables_to_upload["CFSilver"] = cf_silver

    # Candidate scoring is performed once and retained for comparison evidence.
    cf_fuzzy_candidates = run_timed_step(
        "Contracts Finder fuzzy candidate scoring",
        lambda: create_fuzzy_candidates(
            cf_silver,
            "Supplier_Name",
            sponsor_names,
            "CF"
        )
    )

    cf_gold_fuzzy = run_timed_step(
        "Contracts Finder threshold-qualified fuzzy Gold",
        lambda: create_fuzzy_gold(
            cf_silver,
            "Supplier_Name",
            sponsor_lookup,
            sponsor_names,
            "CF",
            candidate_df=cf_fuzzy_candidates
        )
    )
    created_files.append(save_csv(cf_gold_fuzzy, "CF", "GOLD", "FUZ"))
    tables_to_upload["CFGoldFuz"] = cf_gold_fuzzy

    cf_ml_candidates = run_timed_step(
        "Contracts Finder TF-IDF candidate scoring",
        lambda: create_ml_candidates(
            cf_silver,
            "Supplier_Name",
            sponsor_names,
            "CF"
        )
    )

    cf_gold_ml = run_timed_step(
        "Contracts Finder threshold-qualified TF-IDF Gold",
        lambda: create_ml_gold(
            cf_silver,
            "Supplier_Name",
            sponsor_lookup,
            sponsor_names,
            "CF",
            candidate_df=cf_ml_candidates
        )
    )
    created_files.append(save_csv(cf_gold_ml, "CF", "GOLD", "ML"))
    tables_to_upload["CFGoldML"] = cf_gold_ml

    cf_comparison = run_timed_step(
        "Contracts Finder automated validation comparison",
        lambda: create_matching_comparison(
            cf_fuzzy_candidates,
            cf_ml_candidates,
            "CF"
        )
    )

    # Hard stop before SQL if candidate score evidence is incomplete.
    validate_comparison_score_evidence(
        cf_comparison,
        "Contracts Finder"
    )

    created_files.append(save_csv(cf_comparison, "CF", "COMPARISON"))
    tables_to_upload["CFComparison"] = cf_comparison


if RUN_GTR:
    print("\n================ GATEWAY TO RESEARCH PIPELINE ================")

    gtr_bronze = run_timed_step(
        "Gateway to Research API extraction - Bronze",
        fetch_gtr_bronze
    )
    created_files.append(save_csv(gtr_bronze, "GTR", "BRONZE"))
    tables_to_upload["GTRBronze"] = gtr_bronze

    gtr_silver = run_timed_step(
        "Gateway to Research cleaning - Silver",
        lambda: clean_gtr_silver(gtr_bronze)
    )
    created_files.append(save_csv(gtr_silver, "GTR", "SILVER"))
    tables_to_upload["GTRSilver"] = gtr_silver

    gtr_fuzzy_candidates = run_timed_step(
        "Gateway to Research fuzzy candidate scoring",
        lambda: create_fuzzy_candidates(
            gtr_silver,
            "lead_organisation",
            sponsor_names,
            "GTR"
        )
    )

    gtr_gold_fuzzy = run_timed_step(
        "Gateway to Research threshold-qualified fuzzy Gold",
        lambda: create_fuzzy_gold(
            gtr_silver,
            "lead_organisation",
            sponsor_lookup,
            sponsor_names,
            "GTR",
            candidate_df=gtr_fuzzy_candidates
        )
    )
    created_files.append(save_csv(gtr_gold_fuzzy, "GTR", "GOLD", "FUZ"))
    tables_to_upload["GTRGoldFuz"] = gtr_gold_fuzzy

    gtr_ml_candidates = run_timed_step(
        "Gateway to Research TF-IDF candidate scoring",
        lambda: create_ml_candidates(
            gtr_silver,
            "lead_organisation",
            sponsor_names,
            "GTR"
        )
    )

    gtr_gold_ml = run_timed_step(
        "Gateway to Research threshold-qualified TF-IDF Gold",
        lambda: create_ml_gold(
            gtr_silver,
            "lead_organisation",
            sponsor_lookup,
            sponsor_names,
            "GTR",
            candidate_df=gtr_ml_candidates
        )
    )
    created_files.append(save_csv(gtr_gold_ml, "GTR", "GOLD", "ML"))
    tables_to_upload["GTRGoldML"] = gtr_gold_ml

    gtr_comparison = run_timed_step(
        "Gateway to Research automated validation comparison",
        lambda: create_matching_comparison(
            gtr_fuzzy_candidates,
            gtr_ml_candidates,
            "GTR"
        )
    )

    validate_comparison_score_evidence(
        gtr_comparison,
        "Gateway to Research"
    )

    created_files.append(save_csv(gtr_comparison, "GTR", "COMPARISON"))
    tables_to_upload["GTRComparison"] = gtr_comparison


# Save timing information for the extraction, cleaning and matching stages.
runtime_path = save_runtime_log()
if runtime_path:
    created_files.append(runtime_path)

# ==============================================================
# PIPELINE EVIDENCE SUMMARY
# ==============================================================

def show_dataframe_preview(title, df, max_rows=5):
    print("\n" + title)
    if df is None:
        print("No dataframe available.")
        return
    print("Rows:", len(df), "| Columns:", len(df.columns))
    try:
        from IPython.display import display
        display(df.head(max_rows))
    except Exception:
        print(df.head(max_rows).to_string(index=False))


def show_comparison_summary(title, df):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    if df is None or df.empty:
        print("No comparison rows available.")
        return

    print("Total comparison rows:", len(df))

    if "auto_validation_status" in df.columns:
        print("\nAutomated validation status counts:")
        print(
            df["auto_validation_status"]
            .fillna("MISSING")
            .value_counts()
            .to_string()
        )

    if "dashboard_eligible" in df.columns:
        eligible = (
            pd.to_numeric(
                df["dashboard_eligible"],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )
        print("\nDashboard eligible rows:", int(eligible.sum()))
        print(
            "Excluded / not eligible rows:",
            int((eligible == 0).sum())
        )

    if "methods_agree" in df.columns:
        agree = (
            pd.to_numeric(
                df["methods_agree"],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )
        if len(agree) > 0:
            print(
                "Fuzzy and TF-IDF candidate agreement rate:",
                round(float(agree.mean() * 100), 2),
                "%"
            )

    fuzzy_scores = pd.to_numeric(
        df.get("fuzzy_score_percent"),
        errors="coerce"
    )
    ml_scores = pd.to_numeric(
        df.get("ml_score_percent"),
        errors="coerce"
    )

    print(
        "Missing fuzzy candidate scores:",
        int(fuzzy_scores.isna().sum())
    )
    print(
        "Missing ML candidate scores:",
        int(ml_scores.isna().sum())
    )


if SHOW_PRESENTATION_SUMMARY:
    print("\n" + "#" * 70)
    print("PIPELINE EVIDENCE SUMMARY")
    print("#" * 70)
    print("Run mode:", RUN_MODE)

    print("\nFiles created in this run:")
    for file_path in created_files:
        if file_path:
            print("-", file_path)

    if RUN_CONTRACTS_FINDER:
        show_dataframe_preview(
            "Contracts Finder Bronze preview",
            globals().get("cf_bronze"),
            max_rows=3
        )
        show_dataframe_preview(
            "Contracts Finder Silver preview",
            globals().get("cf_silver"),
            max_rows=3
        )
        show_comparison_summary(
            "Contracts Finder automated validation evidence",
            globals().get("cf_comparison")
        )

    if RUN_GTR:
        show_dataframe_preview(
            "Gateway to Research Bronze preview",
            globals().get("gtr_bronze"),
            max_rows=3
        )
        show_dataframe_preview(
            "Gateway to Research Silver preview",
            globals().get("gtr_silver"),
            max_rows=3
        )
        show_comparison_summary(
            "Gateway to Research automated validation evidence",
            globals().get("gtr_comparison")
        )

    print("\nPipeline summary completed.")


Loaded 126,582 unique sponsor organisations.
[TIME] Load Home Office Sponsor Register: 43.59 seconds | SUCCESS
Saved 126,582 rows -> /content/drive/MyDrive/LD6053-dissertation/20260810/HOME_SPONSOR.csv

================ CONTRACTS FINDER PIPELINE ================
Contracts Finder recent publication window: 2026-05-12 to 2026-08-10


Contracts Finder recent award pages:   1%|          | 3/300 [00:10<17:23,  3.51s/it]


Contracts Finder collected at least 300 recent award-supplier rows; stopping API pagination early.
Contracts Finder stable-ID deduplication removed 0 duplicate row(s).
Contracts Finder removed 49 row(s) outside the recent award-date window.
Contracts Finder retained 277 award-supplier row(s) from 2026-05-12 to 2026-08-09.
[TIME] Contracts Finder API extraction - Bronze: 10.56 seconds | SUCCESS
Saved 277 rows -> /content/drive/MyDrive/LD6053-dissertation/20260810/CF_BRONZE.csv
[TIME] Contracts Finder cleaning - Silver: 0.02 seconds | SUCCESS
Saved 277 rows -> /content/drive/MyDrive/LD6053-dissertation/20260810/CF_SILVER.csv


CF fuzzy candidate scoring: 100%|██████████| 222/222 [00:00<00:00, 5260.83it/s]


[TIME] Contracts Finder fuzzy candidate scoring: 9.77 seconds | SUCCESS
[TIME] Contracts Finder threshold-qualified fuzzy Gold: 0.07 seconds | SUCCESS
Saved 98 rows -> /content/drive/MyDrive/LD6053-dissertation/20260810/CF_GOLD_FUZ.csv


...


Contracts Finder automated validation evidence
Total comparison rows: 222

Automated validation status counts:
auto_validation_status
EXCLUDED_LOW_CONFIDENCE         136
ACCEPTED                         53
EXCLUDED_SINGLE_METHOD           29
EXCLUDED_METHOD_DISAGREEMENT      4

Dashboard eligible rows: 53
Excluded / not eligible rows: 169
Fuzzy and TF-IDF candidate agreement rate: 40.54 %
Missing fuzzy candidate scores: 0
Missing ML candidate scores: 0

Gateway to Research Bronze preview
Rows: 250 | Columns: 24


,record_id,project_id,project_url,title,status,grant_reference,grant_category,abstract_text,start_date,end_date,...,org_url,lead_organisation,API_Page,raw_project_json,source_system,medallion_layer,inserted_date,modified_date,matching_method,matching_accuracy_percent
0,1,03B7E12D-8E1E-4159-9079-8E5B5EDA2FBA,http://gtr.ukri.org/api/projects?ref=UKRI4768,Multimodal risk prediction in Traumatic Brain ...,ACTIVE,UKRI4768,Fellowship,None,2026-08-03,2029-08-03,...,http://gtr.ukri.org/api/organisation/8E38A887-...,UNIVERSITY OF CAMBRIDGE,3,"{""id"": ""03B7E12D-8E1E-4159-9079-8E5B5EDA2FBA"",...",GTR,BRONZE,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None
9,2,F397354B-9EEC-4DD8-9504-7FA853589FD3,http://gtr.ukri.org/api/projects?ref=UKRI3509,RNA-protein condensates in neuronal health and...,ACTIVE,UKRI3509,Fellowship,None,2026-07-30,2029-07-30,...,http://gtr.ukri.org/api/organisation/4E4B9EA3-...,UNIVERSITY OF SHEFFIELD,3,"{""id"": ""F397354B-9EEC-4DD8-9504-7FA853589FD3"",...",GTR,BRONZE,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None
15,3,37210B0F-898C-44DA-B384-FF448BB9C13C,http://gtr.ukri.org/api/projects?ref=UKRI4732,Reframing non-CO2 climate mitigation approache...,ACTIVE,UKRI4732,Fellowship,None,2026-07-30,2029-07-30,...,http://gtr.ukri.org/api/organisation/8D60FA62-...,CRANFIELD UNIVERSITY,3,"{""id"": ""37210B0F-898C-44DA-B384-FF448BB9C13C"",...",GTR,BRONZE,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None



Gateway to Research Silver preview
Rows: 250 | Columns: 25


,record_id,project_id,project_url,title,status,grant_reference,grant_category,abstract_text,start_date,end_date,...,lead_organisation,API_Page,raw_project_json,source_system,medallion_layer,inserted_date,modified_date,matching_method,matching_accuracy_percent,lead_organisation_clean
0,1,03B7E12D-8E1E-4159-9079-8E5B5EDA2FBA,http://gtr.ukri.org/api/projects?ref=UKRI4768,Multimodal risk prediction in Traumatic Brain ...,ACTIVE,UKRI4768,Fellowship,None,2026-08-03,2029-08-03,...,UNIVERSITY OF CAMBRIDGE,3,"{""id"": ""03B7E12D-8E1E-4159-9079-8E5B5EDA2FBA"",...",GTR,SILVER,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None,UNIVERSITY OF CAMBRIDGE
8,10,333368C1-0C24-44D3-8732-A7E75AC6732A,http://gtr.ukri.org/api/projects?ref=UKRI3505,Pan-African Frontiers: Reparative Futures and ...,ACTIVE,UKRI3505,Fellowship,None,2026-07-30,2029-07-30,...,SCHOOL OF ORIENTAL AND AFRICAN STUDIES,3,"{""id"": ""333368C1-0C24-44D3-8732-A7E75AC6732A"",...",GTR,SILVER,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None,SCHOOL OF ORIENTAL AND AFRICAN STUDIES
7,16,5AEF23BA-459A-4236-8384-A5EC3818B0CE,http://gtr.ukri.org/api/projects?ref=UKRI3485,Unlearning AI,ACTIVE,UKRI3485,Fellowship,None,2026-07-30,2029-07-30,...,THE OPEN UNIVERSITY,3,"{""id"": ""5AEF23BA-459A-4236-8384-A5EC3818B0CE"",...",GTR,SILVER,2026-08-10 19:16:08,2026-08-10 19:16:08,None,None,THE OPEN UNIVERSITY



Gateway to Research automated validation evidence
Total comparison rows: 141

Automated validation status counts:
auto_validation_status
ACCEPTED                        81
EXCLUDED_LOW_CONFIDENCE         51
EXCLUDED_SINGLE_METHOD           8
EXCLUDED_METHOD_DISAGREEMENT     1

Dashboard eligible rows: 81
Excluded / not eligible rows: 60
Fuzzy and TF-IDF candidate agreement rate: 69.5 %
Missing fuzzy candidate scores: 0
Missing ML candidate scores: 0

Pipeline summary completed.


In [9]:
# SECTION 9 - SQL SERVER UPLOAD

# SAFE SQL NOTE:
# fast_executemany is disabled because pyodbc can create string-buffer truncation
# when long SponsorRoute values are inserted into SQL Server.


# Automated validation evidence is stored in dbo.MatchComparison and dbo.vw_AutomatedValidationEvidence.
# DataLoadAudit stores CF and GTR file timestamps; Home Office and runtime-log CSV timestamps remain visible in Google Drive.

if RUN_SQL_UPLOAD:
    sql_total_start = time.perf_counter()
    import pyodbc
    from sqlalchemy import create_engine, text
    from sqlalchemy.pool import NullPool
    from urllib.parse import quote_plus

    connection_string = (
        "DRIVER={ODBC Driver 18 for SQL Server};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DATABASE};"
        f"UID={SQL_USERNAME};"
        f"PWD={SQL_PASSWORD};"
        "Encrypt=yes;"
        "TrustServerCertificate=yes;"
        "Connection Timeout=60;"
        "APP=LD6053_Colab_Autocommit;"
    )

    # NullPool prevents Colab from leaving pooled SQL sessions behind.
    # AUTOCOMMIT means every DELETE or INSERT statement finishes immediately.
    # use_insertmanyvalues=False prevents SQLAlchemy from rewriting rows into one multi-row INSERT.
    engine = create_engine(
        "mssql+pyodbc:///?odbc_connect=" + quote_plus(connection_string),
        fast_executemany=False,
        use_insertmanyvalues=False,
        isolation_level="AUTOCOMMIT",
        poolclass=NullPool,
        pool_pre_ping=True
    )

    # One timestamp is used for the entire SQL run.
    # SQL history is stored in UTC to match SQL Server SYSUTCDATETIME().
    SQL_RUN_TIMESTAMP = current_timestamp()
    SQL_DAY_START = datetime.combine(SQL_RUN_TIMESTAMP.date(), datetime.min.time())
    SQL_DAY_END = SQL_DAY_START + timedelta(days=1)

    print(
        "SQL daily-history mode: previous UTC dates will be preserved. "
        f"Only rows from {SQL_DAY_START} up to (not including) {SQL_DAY_END} "
        "will be replaced if this notebook is rerun on the same UTC date."
    )

    def open_sql_autocommit_connection():
        connection = pyodbc.connect(connection_string, autocommit=True)
        connection.timeout = 120
        return connection

    def get_sql_log_reuse_wait():
        with open_sql_autocommit_connection() as connection:
            cursor = connection.cursor()
            cursor.execute(
                "SELECT log_reuse_wait_desc FROM sys.databases WHERE database_id = DB_ID();"
            )
            row = cursor.fetchone()
            return str(row[0]) if row else "UNKNOWN"

    def wait_for_sql_log(stage_name, allow_active_seconds=60):
        deadline = time.time() + allow_active_seconds
        while True:
            state = get_sql_log_reuse_wait()
            if state != "ACTIVE_TRANSACTION":
                if state not in ["NOTHING", "CHECKPOINT"]:
                    print(f"SQL log status before {stage_name}: {state}")
                return state

            if time.time() >= deadline:
                raise RuntimeError(
                    "SQL upload stopped safely before changing more rows because the database "
                    "still reports ACTIVE_TRANSACTION. Close/disconnect the session holding the "
                    "transaction or ask the database host to release it, then rerun Section 9."
                )

            print(f"SQL log is held by an active transaction. Waiting before {stage_name}...")
            time.sleep(5)

    def best_effort_checkpoint():
        try:
            with open_sql_autocommit_connection() as connection:
                connection.cursor().execute("CHECKPOINT;")
        except Exception:
            pass

    def is_log_full_error(error):
        message = str(error).upper()
        return "9002" in message or "TRANSACTION LOG" in message and "FULL" in message


    def is_storage_capacity_error(error):
        message = str(error).upper()
        return (
            "1105" in message
            or "PRIMARY FILEGROUP IS FULL" in message
            or "COULD NOT ALLOCATE SPACE" in message
        )


    def database_storage_status():
        """
        Read-only database-file space check.
        Returns allocated/used/free MB inside the current ROWS data files.
        """
        try:
            with open_sql_autocommit_connection() as connection:
                cursor = connection.cursor()
                cursor.execute("""
                    SELECT
                        CAST(SUM(CAST(size AS BIGINT)) * 8.0 / 1024.0 AS DECIMAL(18,2)) AS AllocatedMB,
                        CAST(SUM(CAST(FILEPROPERTY(name, 'SpaceUsed') AS BIGINT)) * 8.0 / 1024.0 AS DECIMAL(18,2)) AS UsedMB,
                        CAST(SUM(CAST(size - FILEPROPERTY(name, 'SpaceUsed') AS BIGINT)) * 8.0 / 1024.0 AS DECIMAL(18,2)) AS FreeInsideFileMB,
                        MAX(CASE WHEN max_size = -1 OR max_size > size THEN 1 ELSE 0 END) AS CanGrow
                    FROM sys.database_files
                    WHERE type = 0;
                """)
                row = cursor.fetchone()
                if not row:
                    return None

                return {
                    "allocated_mb": float(row[0] or 0),
                    "used_mb": float(row[1] or 0),
                    "free_mb": float(row[2] or 0),
                    "can_grow": bool(row[3])
                }
        except Exception as error:
            print(
                "SQL storage preflight could not read database-file usage. "
                "The upload will continue with fail-safe cleanup enabled."
            )
            print("Storage preflight reason:", str(error).splitlines()[0])
            return None


    def print_database_storage_status(label):
        status = database_storage_status()
        if status is None:
            return None

        print(
            f"{label}: allocated={status['allocated_mb']:.2f} MB | "
            f"used={status['used_mb']:.2f} MB | "
            f"free-inside-file={status['free_mb']:.2f} MB | "
            f"can-grow={'YES' if status['can_grow'] else 'NO'}"
        )
        return status

    REQUIRED_TABLES = [
        "DataLoadAudit",
        "DimSponsorOrganisation",
        "Gold_ContractsFinder",
        "Gold_GtRResearch",
        "MatchComparison"
    ]

    def sql_table_exists(conn, table_name):
        return conn.execute(
            text("SELECT CASE WHEN OBJECT_ID(:table_name, 'U') IS NULL THEN 0 ELSE 1 END"),
            {"table_name": f"dbo.{table_name}"}
        ).scalar() == 1

    def clean_target_value(value):
        if pd.isna(value):
            return None
        return value

    def find_column(df, candidates):
        if df is None or df.empty:
            return None

        def norm(x):
            return re.sub(r"[^a-z0-9]", "", str(x).lower())

        lookup = {norm(c): c for c in df.columns}
        for candidate in candidates:
            direct = candidate
            if direct in df.columns:
                return direct
            key = norm(candidate)
            if key in lookup:
                return lookup[key]
        return None

    def series_or_none(df, candidates):
        col = find_column(df, candidates)
        if col is None:
            return pd.Series([None] * len(df), index=df.index)
        return df[col]

    def method_to_sql(value):
        value = str(value).strip().upper()
        if "EXACT" in value:
            return "Exact"
        if "ML" in value or "TFIDF" in value or "COSINE" in value:
            return "ML"
        return "Fuzzy"

    def prepare_gold_for_target(df_gold, source_system, sponsor_key_map):
        if df_gold is None or df_gold.empty:
            if source_system == "CF":
                return pd.DataFrame(columns=[
                    "SponsorKey", "Contract_Title", "Contract_Value", "Supplier_Name",
                    "Award_Date", "matched_sponsor", "Town_City", "Route",
                    "MatchMethod", "MatchingAccuracyPercent", "InsertedDate", "ModifiedDate"
                ])
            return pd.DataFrame(columns=[
                "SponsorKey", "project_id", "title", "status", "start_date",
                "lead_organisation", "matched_sponsor", "MatchMethod",
                "MatchingAccuracyPercent", "InsertedDate", "ModifiedDate"
            ])

        df = df_gold.copy()
        official_col = find_column(df, [
            "sponsor_Organisation Name", "sponsor_Organisation_Name",
            "sponsor_organisation_name", "matched_sponsor",
            "Organisation Name", "Organisation_Name"
        ])
        town_col = find_column(df, ["sponsor_Town/City", "sponsor_Town_City", "Town_City", "TownCity", "Town/City"])
        route_col = find_column(df, ["sponsor_Route", "Route", "SponsorRoute"])
        match_method_col = find_column(df, ["matching_method", "MatchMethod", "method"])
        score_col = find_column(df, [
            "matching_accuracy_percent", "MatchingAccuracyPercent",
            "matching_confidence_percent", "match_score", "score_percent",
            "fuzzy_score_percent", "ml_score_percent", "FuzzyScorePercent", "MLScorePercent"
        ])

        if official_col is not None:
            matched_sponsor = df[official_col].fillna(df.get("matched_sponsor_clean", ""))
        else:
            matched_sponsor = df.get("matched_sponsor_clean", pd.Series([None] * len(df), index=df.index))

        common = pd.DataFrame(index=df.index)
        common["matched_sponsor"] = matched_sponsor.astype(str).replace({"nan": None, "": None})
        common["SponsorKey"] = common["matched_sponsor"].apply(
            lambda value: sponsor_key_map.get(str(value).strip().casefold())
            if value is not None and str(value).strip() else None
        )
        common["MatchMethod"] = series_or_none(df, [match_method_col if match_method_col else "matching_method"]).apply(method_to_sql)

        score_values = pd.to_numeric(
            series_or_none(df, [score_col if score_col else "matching_accuracy_percent"]),
            errors="coerce"
        )

        source_clean = series_or_none(df, ["source_organisation_clean"]).astype(str)
        matched_clean = series_or_none(df, ["matched_sponsor_clean"]).astype(str)
        exact_like = (source_clean == matched_clean) | (common["MatchMethod"] == "Exact")
        score_values = score_values.mask(score_values.isna() & exact_like, 100.0)

        common["MatchingAccuracyPercent"] = score_values
        common["InsertedDate"] = SQL_RUN_TIMESTAMP
        common["ModifiedDate"] = SQL_RUN_TIMESTAMP

        if source_system == "CF":
            out = pd.DataFrame(index=df.index)
            out["SponsorKey"] = common["SponsorKey"]
            out["Contract_Title"] = series_or_none(df, ["Contract_Title"])
            out["Contract_Value"] = pd.to_numeric(series_or_none(df, ["Contract_Value"]), errors="coerce")
            out["Supplier_Name"] = series_or_none(df, ["Supplier_Name"])
            out["Award_Date"] = series_or_none(df, ["Award_Date"])
            out["matched_sponsor"] = common["matched_sponsor"]
            out["Town_City"] = series_or_none(df, [town_col if town_col else "Town_City"])
            out["Route"] = series_or_none(df, [route_col if route_col else "Route"])
            out["MatchMethod"] = common["MatchMethod"]
            out["MatchingAccuracyPercent"] = common["MatchingAccuracyPercent"]
            out["InsertedDate"] = common["InsertedDate"]
            out["ModifiedDate"] = common["ModifiedDate"]
            return out

        out = pd.DataFrame(index=df.index)
        out["SponsorKey"] = common["SponsorKey"]
        out["project_id"] = series_or_none(df, ["project_id"])
        out["title"] = series_or_none(df, ["title"])
        out["status"] = series_or_none(df, ["status"])
        out["start_date"] = series_or_none(df, ["start_date"])
        out["lead_organisation"] = series_or_none(df, ["lead_organisation"])
        out["matched_sponsor"] = common["matched_sponsor"]
        out["MatchMethod"] = common["MatchMethod"]
        out["MatchingAccuracyPercent"] = common["MatchingAccuracyPercent"]
        out["InsertedDate"] = common["InsertedDate"]
        out["ModifiedDate"] = common["ModifiedDate"]
        return out

    def build_sponsor_dimension(gold_frames):
        rows = []
        for df in gold_frames:
            if df is None or df.empty:
                continue

            official_col = find_column(df, [
                "sponsor_Organisation Name", "sponsor_Organisation_Name",
                "sponsor_organisation_name", "matched_sponsor"
            ])
            town_col = find_column(df, ["sponsor_Town/City", "sponsor_Town_City", "Town_City", "TownCity"])
            route_col = find_column(df, ["sponsor_Route", "Route", "SponsorRoute"])

            if official_col is None:
                official = df.get("matched_sponsor_clean", pd.Series([None] * len(df), index=df.index))
            else:
                official = df[official_col].fillna(df.get("matched_sponsor_clean", ""))

            temp = pd.DataFrame({
                "MatchedSponsor": official,
                "TownCity": series_or_none(df, [town_col if town_col else "TownCity"]),
                "SponsorRoute": series_or_none(df, [route_col if route_col else "SponsorRoute"])
            })
            rows.append(temp)

        if not rows:
            return pd.DataFrame(columns=["MatchedSponsor", "TownCity", "SponsorRoute", "InsertedDate", "ModifiedDate"])

        dim = pd.concat(rows, ignore_index=True)
        dim["MatchedSponsor"] = dim["MatchedSponsor"].astype(str).replace({"nan": None, "": None})
        dim = dim.dropna(subset=["MatchedSponsor"])
        dim["_SponsorNameKey"] = dim["MatchedSponsor"].astype(str).str.strip().str.casefold()
        dim = dim.drop_duplicates(subset=["_SponsorNameKey"]).drop(columns=["_SponsorNameKey"])
        dim["InsertedDate"] = SQL_RUN_TIMESTAMP
        dim["ModifiedDate"] = SQL_RUN_TIMESTAMP
        return dim[["MatchedSponsor", "TownCity", "SponsorRoute", "InsertedDate", "ModifiedDate"]]

    def prepare_comparison_for_target(df_comparison, source_system):
        if df_comparison is None or df_comparison.empty:
            return pd.DataFrame(columns=[
                "SourceSystem", "SourceRecordKey", "SourceOrganisationName",
                "FuzzyMatchedSponsor", "FuzzyScorePercent", "MLMatchedSponsor",
                "MLScorePercent", "MethodsAgree", "RecommendedMethod",
                "RecommendedMatchedSponsor", "RecommendedScorePercent",
                "AutoValidationStatus", "AutoValidationReason", "IsDashboardEligible",
                "FuzzyCorrect", "MLCorrect", "ValidationNotes",
                "InsertedDate", "ModifiedDate"
            ])

        df = df_comparison.copy()
        source_name = (
            "ContractsFinder"
            if source_system == "CF"
            else "GtRResearch"
        )

        source_org = series_or_none(
            df,
            [
                "source_organisation_clean",
                "SourceOrganisationName",
                "SourceRecordKey"
            ]
        )

        # Raw candidate names are retained in the comparison CSV.
        fuzzy_candidate = series_or_none(
            df,
            [
                "fuzzy_match",
                "FuzzyMatchedSponsor",
                "fuzzy_matched_sponsor"
            ]
        )
        ml_candidate = series_or_none(
            df,
            [
                "ml_match",
                "MLMatchedSponsor",
                "ml_matched_sponsor"
            ]
        )

        # Scores are retained even when a method is below its threshold.
        fuzzy_score = pd.to_numeric(
            series_or_none(
                df,
                [
                    "fuzzy_score_percent",
                    "FuzzyScorePercent",
                    "fuzzy_score",
                    "fuzzy_match_score"
                ]
            ),
            errors="coerce"
        )
        ml_score = pd.to_numeric(
            series_or_none(
                df,
                [
                    "ml_score_percent",
                    "MLScorePercent",
                    "ml_score",
                    "ml_match_score"
                ]
            ),
            errors="coerce"
        )

        source_as_text = source_org.astype(str)

        fuzzy_score = fuzzy_score.mask(
            fuzzy_score.isna()
            & fuzzy_candidate.notna()
            & (
                source_as_text
                == fuzzy_candidate.astype(str)
            ),
            100.0
        )

        ml_score = ml_score.mask(
            ml_score.isna()
            & ml_candidate.notna()
            & (
                source_as_text
                == ml_candidate.astype(str)
            ),
            100.0
        )

        # Use the explicit threshold-qualified flags generated in Section 5.
        fuzzy_found = pd.to_numeric(
            series_or_none(
                df,
                ["fuzzy_found_match"]
            ),
            errors="coerce"
        )

        ml_found = pd.to_numeric(
            series_or_none(
                df,
                ["ml_found_match"]
            ),
            errors="coerce"
        )

        # Fallback for older comparison dataframes only.
        if fuzzy_found.isna().all():
            fuzzy_found = (
                fuzzy_candidate.notna()
                & (
                    fuzzy_score
                    >= float(FUZZY_THRESHOLD)
                )
            ).astype(int)
        else:
            fuzzy_found = fuzzy_found.fillna(0).astype(int)

        if ml_found.isna().all():
            ml_found = (
                ml_candidate.notna()
                & (
                    ml_score
                    >= float(ML_THRESHOLD_PERCENT)
                )
            ).astype(int)
        else:
            ml_found = ml_found.fillna(0).astype(int)

        # SQL match-name fields keep their historical meaning:
        # "method produced a threshold-qualified match".
        # The score columns still keep the low-score candidate evidence.
        fuzzy_match_for_sql = fuzzy_candidate.where(
            fuzzy_found == 1,
            None
        )
        ml_match_for_sql = ml_candidate.where(
            ml_found == 1,
            None
        )

        methods_agree_source = pd.to_numeric(
            series_or_none(
                df,
                ["methods_agree", "MethodsAgree"]
            ),
            errors="coerce"
        )

        if methods_agree_source.isna().all():
            methods_agree = (
                fuzzy_candidate.fillna("").astype(str)
                == ml_candidate.fillna("").astype(str)
            ) & (
                fuzzy_candidate.notna()
                & ml_candidate.notna()
            )
            methods_agree = methods_agree.astype(int)
        else:
            methods_agree = (
                methods_agree_source
                .fillna(0)
                .astype(int)
            )

        recommended_method = series_or_none(
            df,
            [
                "recommended_method",
                "RecommendedMethod"
            ]
        )

        # SQL-safe method labels.
        def sql_safe_recommended_method(value):
            if pd.isna(value):
                return None

            v = str(value).strip()
            lower_v = v.lower()

            if v in ["Fuzzy", "ML"]:
                return v

            if lower_v in [
                "exact",
                "fuzzy+ml",
                "fuzzy + ml",
                "both",
                "agreement",
                "methodsagree"
            ]:
                return "Fuzzy"

            if "ml" in lower_v and "fuzzy" not in lower_v:
                return "ML"

            if "fuzzy" in lower_v or "exact" in lower_v:
                return "Fuzzy"

            return None

        recommended_method = (
            recommended_method
            .apply(sql_safe_recommended_method)
        )

        recommended_match = series_or_none(
            df,
            [
                "recommended_match",
                "RecommendedMatchedSponsor"
            ]
        )

        recommended_score = pd.to_numeric(
            series_or_none(
                df,
                [
                    "recommended_score_percent",
                    "RecommendedScorePercent"
                ]
            ),
            errors="coerce"
        )

        # Candidate scores are now available for excluded rows as well.
        recommended_score = recommended_score.fillna(
            pd.concat(
                [fuzzy_score, ml_score],
                axis=1
            ).max(axis=1)
        )
        recommended_score = (
            recommended_score
            .fillna(0)
            .clip(lower=0, upper=100)
            .round(2)
        )

        auto_status = series_or_none(
            df,
            [
                "auto_validation_status",
                "AutoValidationStatus"
            ]
        )
        auto_reason = series_or_none(
            df,
            [
                "auto_validation_reason",
                "AutoValidationReason"
            ]
        )
        dashboard_eligible = (
            pd.to_numeric(
                series_or_none(
                    df,
                    [
                        "dashboard_eligible",
                        "IsDashboardEligible"
                    ]
                ),
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )

        # Final hard check before creating SQL rows.
        bad_fuzzy = (
            fuzzy_candidate.notna()
            & fuzzy_score.isna()
        )
        bad_ml = (
            ml_candidate.notna()
            & ml_score.isna()
        )

        if bad_fuzzy.any() or bad_ml.any():
            raise RuntimeError(
                "SAFE STOP: comparison candidate scores are incomplete. "
                f"Missing fuzzy scores: {int(bad_fuzzy.sum())}; "
                f"missing ML scores: {int(bad_ml.sum())}. "
                "No incomplete MatchComparison rows will be uploaded."
            )

        out = pd.DataFrame({
            "SourceSystem": source_name,
            "SourceRecordKey": source_org,
            "SourceOrganisationName": source_org,
            "FuzzyMatchedSponsor": fuzzy_match_for_sql,
            "FuzzyScorePercent": fuzzy_score,
            "MLMatchedSponsor": ml_match_for_sql,
            "MLScorePercent": ml_score,
            "MethodsAgree": methods_agree,
            "RecommendedMethod": recommended_method,
            "RecommendedMatchedSponsor": recommended_match,
            "RecommendedScorePercent": recommended_score,
            "AutoValidationStatus": auto_status,
            "AutoValidationReason": auto_reason,
            "IsDashboardEligible": dashboard_eligible,
            "FuzzyCorrect": None,
            "MLCorrect": None,
            "ValidationNotes": auto_reason,
            "InsertedDate": SQL_RUN_TIMESTAMP,
            "ModifiedDate": SQL_RUN_TIMESTAMP
        })

        return out


    def filter_gold_by_dashboard_eligibility(df_gold, comparison_df):
        if df_gold is None or df_gold.empty:
            return df_gold
        if comparison_df is None or comparison_df.empty:
            return df_gold.head(0).copy()
        if "dashboard_eligible" not in comparison_df.columns:
            return df_gold

        eligible_orgs = set(
            comparison_df.loc[
                comparison_df["dashboard_eligible"].astype(str).isin(["1", "True", "true"]),
                "source_organisation_clean"
            ].dropna().astype(str)
        )

        if "source_organisation_clean" not in df_gold.columns:
            return df_gold

        return df_gold[df_gold["source_organisation_clean"].astype(str).isin(eligible_orgs)].copy()


    def build_recommended_gold(fuzzy_df, ml_df, comparison_df):
        """Build one dashboard-ready Gold dataset using accepted automated validation results only."""
        if comparison_df is None or comparison_df.empty:
            return pd.DataFrame()
        if "dashboard_eligible" not in comparison_df.columns:
            return pd.DataFrame()

        accepted = comparison_df[comparison_df["dashboard_eligible"].astype(str).isin(["1", "True", "true"])].copy()
        if accepted.empty:
            return pd.DataFrame()

        # Use fuzzy rows as the main record source because accepted records require method agreement or exact match.
        # Fall back to ML rows if fuzzy rows are not available.
        base = fuzzy_df.copy() if fuzzy_df is not None and not fuzzy_df.empty else pd.DataFrame()
        if base.empty and ml_df is not None and not ml_df.empty:
            base = ml_df.copy()
        if base.empty or "source_organisation_clean" not in base.columns:
            return pd.DataFrame()

        accepted_cols = accepted[[
            "source_organisation_clean",
            "recommended_method",
            "recommended_match",
            "recommended_score_percent",
            "auto_validation_status",
            "auto_validation_reason",
            "dashboard_eligible"
        ]].copy()

        out = base.merge(accepted_cols, on="source_organisation_clean", how="inner")
        if out.empty:
            return out

        out["matched_sponsor_clean"] = out["recommended_match"]
        out["matching_method"] = out["recommended_method"]
        out["matching_accuracy_percent"] = out["recommended_score_percent"]
        return out


    def build_audit_dataframe(file_paths):
        rows = []
        # Only these two SourceSystem values are uploaded to DataLoadAudit.
        # Some database versions have a CHECK constraint that rejects other values.
        source_lookup = {
            "CF": "ContractsFinder",
            "GTR": "GtRResearch"
        }
        layer_lookup = {
            "BRONZE": "Bronze",
            "SILVER": "Silver",
            "GOLD": "Gold",
            "COMPARISON": "Comparison"
        }
        method_lookup = {"FUZ": "Fuzzy", "ML": "ML"}

        for file_path in file_paths:
            file_name = os.path.basename(file_path)
            name_without_ext = os.path.splitext(file_name)[0]
            parts = name_without_ext.split("_")

            source_code = parts[0] if len(parts) > 0 else ""
            layer_code = parts[1] if len(parts) > 1 else ""
            method_code = parts[2] if len(parts) > 2 else None

            # HOME_SPONSOR and LD6053_RUNTIME_LOG are kept in Google Drive as evidence,
            # but they are not inserted into DataLoadAudit to avoid SQL CHECK constraint errors.
            if source_code not in source_lookup:
                continue

            try:
                row_count = len(pd.read_csv(file_path, low_memory=False))
            except Exception:
                row_count = 0

            audit_row = {
                "RunDate": RUN_DATE,
                "FileName": file_name,
                "FilePath": file_path,
                "SourceSystem": source_lookup[source_code],
                "LayerName": layer_lookup.get(layer_code, layer_code.title()),
                "MatchMethod": method_lookup.get(method_code, None),
                "RowsLoaded": int(row_count),
                "ModifiedDate": SQL_RUN_TIMESTAMP
            }

            audit_meta = sql_table_metadata("DataLoadAudit")
            audit_columns = set(
                audit_meta["COLUMN_NAME"].astype(str)
            )

            if "InsertedDate" in audit_columns:
                audit_row["InsertedDate"] = SQL_RUN_TIMESTAMP
            elif "LoadedAtUTC" in audit_columns:
                audit_row["LoadedAtUTC"] = SQL_RUN_TIMESTAMP

            rows.append(audit_row)

        return pd.DataFrame(rows)

    def sql_table_metadata(table_name):
        with engine.connect() as conn:
            rows = conn.execute(
                text("""
                    SELECT COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH
                    FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = 'dbo' AND TABLE_NAME = :table_name
                    ORDER BY ORDINAL_POSITION
                """),
                {"table_name": table_name}
            ).fetchall()
        return pd.DataFrame(rows, columns=["COLUMN_NAME", "DATA_TYPE", "CHARACTER_MAXIMUM_LENGTH"])


    def fit_dataframe_to_sql_table(table_name, df):
        """Keep only target columns and trim long text to the SQL column size."""
        df = prepare_for_sql(df)
        meta = sql_table_metadata(table_name)

        if meta.empty:
            return df

        target_columns = meta["COLUMN_NAME"].tolist()
        dropped_columns = [c for c in df.columns if c not in target_columns]
        keep_columns = [c for c in target_columns if c in df.columns]

        if dropped_columns:
            print(f"Columns not found in dbo.{table_name} and not uploaded:", dropped_columns)

        df = df[keep_columns].copy()

        for _, row in meta.iterrows():
            col = row["COLUMN_NAME"]
            data_type = str(row["DATA_TYPE"]).lower()
            max_len = row["CHARACTER_MAXIMUM_LENGTH"]

            if col not in df.columns:
                continue

            if data_type in ["varchar", "nvarchar", "char", "nchar"] and pd.notna(max_len):
                max_len = int(max_len)
                if max_len > 0:
                    df[col] = df[col].apply(
                        lambda x: x[:max_len] if isinstance(x, str) and len(x) > max_len else x
                    )

        return df


    def python_sql_value(value):
        """Convert pandas/numpy values into values accepted by pyodbc."""
        if value is None:
            return None
        try:
            if pd.isna(value):
                return None
        except Exception:
            pass
        if isinstance(value, pd.Timestamp):
            return value.to_pydatetime()
        if isinstance(value, np.generic):
            return value.item()
        return value


    def insert_dataframe(table_name, df):
        """Insert rows using one autocommitted SQL statement per row.

        The page count changes the number of iterations, not the transaction size.
        Therefore a quick, page-limited or full API run uses the same safe SQL behaviour.
        """
        step_start = time.perf_counter()

        try:
            if df is None or df.empty:
                print(f"No rows to insert into dbo.{table_name}")
                log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", 0, "No rows to insert")
                return

            df = fit_dataframe_to_sql_table(table_name, df)

            if df.empty:
                print(f"No matching columns/rows to insert into dbo.{table_name}")
                log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", 0, "No matching rows/columns")
                return

            wait_for_sql_log(f"inserting dbo.{table_name}")

            columns = list(df.columns)
            quoted_columns = ", ".join(f"[{column}]" for column in columns)
            placeholders = ", ".join("?" for _ in columns)
            insert_sql = f"INSERT INTO dbo.[{table_name}] ({quoted_columns}) VALUES ({placeholders});"
            total_rows = len(df)

            connection = open_sql_autocommit_connection()
            cursor = connection.cursor()

            try:
                for row_number, row in enumerate(df.itertuples(index=False, name=None), start=1):
                    values = tuple(python_sql_value(value) for value in row)
                    attempts = 0

                    while True:
                        try:
                            cursor.execute(insert_sql, values)
                            break
                        except Exception as error:
                            attempts += 1
                            if not is_log_full_error(error) or attempts > SQL_RETRY_LIMIT:
                                raise

                            try:
                                cursor.close()
                                connection.close()
                            except Exception:
                                pass

                            best_effort_checkpoint()
                            print(
                                f"SQL log-full warning while inserting dbo.{table_name} row "
                                f"{row_number:,}. Waiting {SQL_RETRY_WAIT_SECONDS} seconds "
                                f"before retry {attempts}/{SQL_RETRY_LIMIT}."
                            )
                            time.sleep(SQL_RETRY_WAIT_SECONDS)
                            wait_for_sql_log(f"retrying dbo.{table_name}")
                            connection = open_sql_autocommit_connection()
                            cursor = connection.cursor()

                    if row_number % SQL_PROGRESS_EVERY == 0 or row_number == total_rows:
                        print(f"dbo.{table_name}: inserted {row_number:,} of {total_rows:,} rows")
                        best_effort_checkpoint()

            finally:
                try:
                    cursor.close()
                except Exception:
                    pass
                try:
                    connection.close()
                except Exception:
                    pass

            print(f"Inserted {total_rows:,} rows into dbo.{table_name}")
            log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "SUCCESS", len(df), "Autocommit row-by-row")

        except Exception as e:
            log_runtime_step(f"SQL insert dbo.{table_name}", step_start, "FAILED", None, str(e))
            save_runtime_log()
            engine.dispose()
            raise


    # Prepare and validate the completed pipeline outputs before replacing today's SQL rows.
    # This prevents an empty or failed pipeline from removing today's existing reporting rows.
    cf_comparison_raw = globals().get("cf_comparison", pd.DataFrame())
    gtr_comparison_raw = globals().get("gtr_comparison", pd.DataFrame())

    cf_gold_sql = build_recommended_gold(
        globals().get("cf_gold_fuzzy", pd.DataFrame()),
        globals().get("cf_gold_ml", pd.DataFrame()),
        cf_comparison_raw
    )
    gtr_gold_sql = build_recommended_gold(
        globals().get("gtr_gold_fuzzy", pd.DataFrame()),
        globals().get("gtr_gold_ml", pd.DataFrame()),
        gtr_comparison_raw
    )

    sponsor_dim = build_sponsor_dimension([cf_gold_sql, gtr_gold_sql])

    if sponsor_dim.empty:
        raise RuntimeError(
            "SQL upload stopped before changing today's SQL rows because no accepted sponsor organisations were prepared."
        )

    if cf_comparison_raw.empty and gtr_comparison_raw.empty:
        raise RuntimeError(
            "SQL upload stopped before changing today's SQL rows because no comparison evidence was prepared."
        )

    print("\nSQL preflight passed.")
    print("Prepared sponsor organisations:", f"{len(sponsor_dim):,}")
    print("Prepared CF comparison organisations:", f"{len(cf_comparison_raw):,}")
    print("Prepared GTR comparison organisations:", f"{len(gtr_comparison_raw):,}")

    def get_daily_timestamp_column(table_name):
        """Return the insertion timestamp column used by a daily-history table."""
        meta = sql_table_metadata(table_name)
        if meta.empty:
            raise RuntimeError(f"Could not read SQL metadata for dbo.{table_name}.")

        columns = set(meta["COLUMN_NAME"].astype(str))

        # Newer project tables use InsertedDate.
        # Older DataLoadAudit versions may use LoadedAtUTC.
        for candidate in ["InsertedDate", "LoadedAtUTC"]:
            if candidate in columns:
                return candidate

        raise RuntimeError(
            f"SAFE STOP: dbo.{table_name} has no InsertedDate or LoadedAtUTC column. "
            "No rows were deleted because the notebook cannot identify only the current day's data."
        )


    def delete_current_day_rows_autocommit(table_name, timestamp_column):
        """Delete only the current UTC day's rows, in small autocommitted batches."""
        wait_for_sql_log(f"replacing today's rows in dbo.{table_name}")
        total_deleted = 0

        connection = open_sql_autocommit_connection()
        cursor = connection.cursor()

        delete_sql = (
            f"DELETE TOP ({int(SQL_DELETE_CHUNK_SIZE)}) "
            f"FROM dbo.[{table_name}] "
            f"WHERE [{timestamp_column}] >= ? AND [{timestamp_column}] < ?;"
        )

        try:
            while True:
                attempts = 0

                while True:
                    try:
                        cursor.execute(delete_sql, SQL_DAY_START, SQL_DAY_END)
                        deleted_now = max(int(cursor.rowcount or 0), 0)
                        break

                    except Exception as error:
                        attempts += 1

                        if not is_log_full_error(error) or attempts > SQL_RETRY_LIMIT:
                            raise

                        try:
                            cursor.close()
                            connection.close()
                        except Exception:
                            pass

                        best_effort_checkpoint()
                        print(
                            f"SQL log-full warning while replacing today's rows in "
                            f"dbo.{table_name}. Waiting {SQL_RETRY_WAIT_SECONDS} seconds "
                            f"before retry {attempts}/{SQL_RETRY_LIMIT}."
                        )
                        time.sleep(SQL_RETRY_WAIT_SECONDS)
                        wait_for_sql_log(f"retrying daily cleanup of dbo.{table_name}")

                        connection = open_sql_autocommit_connection()
                        cursor = connection.cursor()

                total_deleted += deleted_now

                if deleted_now == 0:
                    break

                if total_deleted % SQL_PROGRESS_EVERY == 0:
                    print(
                        f"dbo.{table_name}: removed {total_deleted:,} row(s) "
                        "from the current UTC date only"
                    )
                    best_effort_checkpoint()

        finally:
            try:
                cursor.close()
            except Exception:
                pass
            try:
                connection.close()
            except Exception:
                pass

        print(
            f"dbo.{table_name}: replaced-day cleanup removed "
            f"{total_deleted:,} row(s) from {SQL_DAY_START.date()} only"
        )
        return total_deleted


    # ------------------------------------------------------------------
    # DAILY HISTORY + PRESENTATION-SAFE SQL ORCHESTRATION
    #
    # Rules:
    #   1. Previous dates are never deleted.
    #   2. Only this run's UTC date can be replaced.
    #   3. If SQL storage/network fails after writes begin, today's partial
    #      rows are automatically removed again.
    #   4. Previous completed dates remain untouched.
    #   5. Capacity errors are reported without leaving a red traceback.
    # ------------------------------------------------------------------
    daily_history_tables = [
        "MatchComparison",
        "Gold_ContractsFinder",
        "Gold_GtRResearch",
        "DataLoadAudit"
    ]

    SQL_UPLOAD_COMPLETED = False
    SQL_UPLOAD_SAFE_STOP = False
    SQL_UPLOAD_ERROR = None

    def current_day_count(table_name, timestamp_column):
        with open_sql_autocommit_connection() as connection:
            cursor = connection.cursor()
            cursor.execute(
                f"""
                SELECT COUNT(*)
                FROM dbo.[{table_name}]
                WHERE [{timestamp_column}] >= ?
                  AND [{timestamp_column}] < ?;
                """,
                SQL_DAY_START,
                SQL_DAY_END
            )
            row = cursor.fetchone()
            return int(row[0] if row else 0)


    def delete_unreferenced_current_day_sponsors():
        """
        Remove only sponsor-dimension rows inserted on this UTC date that are
        not referenced by ANY Gold row from ANY date.
        """
        total_deleted = 0
        connection = open_sql_autocommit_connection()
        cursor = connection.cursor()

        sql = (
            f"DELETE TOP ({int(SQL_DELETE_CHUNK_SIZE)}) "
            "FROM dbo.DimSponsorOrganisation "
            "WHERE InsertedDate >= ? AND InsertedDate < ? "
            "AND NOT EXISTS ("
            "    SELECT 1 FROM dbo.Gold_ContractsFinder cf "
            "    WHERE cf.SponsorKey = dbo.DimSponsorOrganisation.SponsorKey"
            ") "
            "AND NOT EXISTS ("
            "    SELECT 1 FROM dbo.Gold_GtRResearch gtr "
            "    WHERE gtr.SponsorKey = dbo.DimSponsorOrganisation.SponsorKey"
            ");"
        )

        try:
            while True:
                cursor.execute(sql, SQL_DAY_START, SQL_DAY_END)
                deleted_now = max(int(cursor.rowcount or 0), 0)
                total_deleted += deleted_now
                if deleted_now == 0:
                    break
        finally:
            try:
                cursor.close()
            except Exception:
                pass
            try:
                connection.close()
            except Exception:
                pass

        if total_deleted:
            print(
                f"dbo.DimSponsorOrganisation: removed {total_deleted:,} "
                "unreferenced sponsor row(s) created on the current UTC date."
            )
        return total_deleted


    def cleanup_current_day_after_failed_upload(timestamp_columns):
        """
        Best-effort recovery after any SQL write failure.
        Deletes CURRENT UTC DATE ONLY from daily tables.
        Previous dates are outside every WHERE condition.
        """
        print("\nSAFE RECOVERY: removing partial current-day SQL rows only.")
        cleanup_errors = []

        # Child/detail tables first.
        for table_name in [
            "MatchComparison",
            "Gold_ContractsFinder",
            "Gold_GtRResearch",
            "DataLoadAudit"
        ]:
            try:
                delete_current_day_rows_autocommit(
                    table_name,
                    timestamp_columns[table_name]
                )
            except Exception as cleanup_error:
                cleanup_errors.append(
                    f"{table_name}: {str(cleanup_error).splitlines()[0]}"
                )

        try:
            delete_unreferenced_current_day_sponsors()
        except Exception as cleanup_error:
            cleanup_errors.append(
                "DimSponsorOrganisation: "
                + str(cleanup_error).splitlines()[0]
            )

        if cleanup_errors:
            print("SAFE RECOVERY encountered cleanup warnings:")
            for item in cleanup_errors:
                print("-", item)
        else:
            print(
                "SAFE RECOVERY COMPLETED: previous dates are preserved and "
                "no partial current-day fact/comparison/audit rows remain."
            )


    try:
        with engine.connect() as conn:
            missing = [
                table_name
                for table_name in REQUIRED_TABLES
                if not sql_table_exists(conn, table_name)
            ]

        if missing:
            raise RuntimeError(
                "Required SQL tables are missing: "
                + ", ".join(missing)
            )

        daily_timestamp_columns = {
            table_name: get_daily_timestamp_column(table_name)
            for table_name in daily_history_tables
        }

        print("\nDaily-history timestamp columns:")
        for table_name, column_name in daily_timestamp_columns.items():
            print(f"  dbo.{table_name}: {column_name}")

        # Build everything before changing SQL.
        with engine.connect() as conn:
            existing_sponsor_df = pd.read_sql(
                "SELECT SponsorKey, MatchedSponsor "
                "FROM dbo.DimSponsorOrganisation;",
                conn
            )

        existing_name_keys = set(
            existing_sponsor_df["MatchedSponsor"]
            .dropna()
            .astype(str)
            .str.strip()
            .str.casefold()
        )

        sponsor_dim_to_insert = sponsor_dim[
            ~sponsor_dim["MatchedSponsor"]
            .astype(str)
            .str.strip()
            .str.casefold()
            .isin(existing_name_keys)
        ].copy()

        # We need sponsor keys for the Gold tables. Existing names can be mapped now.
        existing_sponsor_key_map = {
            str(name).strip().casefold(): key
            for name, key in zip(
                existing_sponsor_df["MatchedSponsor"],
                existing_sponsor_df["SponsorKey"]
            )
            if pd.notna(name)
        }

        # Audit rows can be prepared before any database writes.
        audit_df = build_audit_dataframe(created_files)

        # Show database storage before any same-day replacement.
        before_status = print_database_storage_status(
            "SQL storage before same-day replacement"
        )

        # If a previous attempt today left partial rows, remove only today's
        # rows before retrying. This is also the normal same-day replacement rule.
        existing_today_counts = {
            table_name: current_day_count(
                table_name,
                daily_timestamp_columns[table_name]
            )
            for table_name in daily_history_tables
        }

        print("\nExisting current-day SQL rows before replacement:")
        for table_name, row_count in existing_today_counts.items():
            print(f"  {table_name}: {row_count:,}")

        clear_start = time.perf_counter()
        wait_for_sql_log(
            "starting current-day SQL replacement",
            allow_active_seconds=60
        )

        for table_name in daily_history_tables:
            delete_current_day_rows_autocommit(
                table_name,
                daily_timestamp_columns[table_name]
            )

        # Any sponsor rows left by a failed attempt today can now be removed
        # if no Gold row from any date references them.
        delete_unreferenced_current_day_sponsors()

        log_runtime_step(
            "SQL replace current UTC day",
            clear_start,
            "SUCCESS",
            None,
            f"Previous dates preserved; replaced {SQL_DAY_START.date()} only"
        )

        after_cleanup_status = print_database_storage_status(
            "SQL storage after current-day cleanup"
        )

        # A database with virtually no free internal data-file space and no
        # growth permission cannot accept the prepared upload. Stop before
        # inserting anything instead of creating another partial day.
        if (
            SQL_STORAGE_PREFLIGHT
            and after_cleanup_status is not None
            and after_cleanup_status["free_mb"] < float(SQL_STORAGE_MIN_FREE_MB)
            and not after_cleanup_status["can_grow"]
        ):
            SQL_UPLOAD_SAFE_STOP = True
            SQL_UPLOAD_ERROR = (
                "SQL data file has insufficient free space and cannot grow."
            )
            print(
                "\nSQL SAFE STOP: database storage is full/near-full. "
                "No current-day rows were inserted after cleanup."
            )
            print(
                "Previous dates remain intact. CSV outputs for this run are "
                "already saved in Google Drive."
            )

        if not SQL_UPLOAD_SAFE_STOP:
            print(
                f"\nExisting sponsor organisations reused: "
                f"{len(sponsor_dim) - len(sponsor_dim_to_insert):,}"
            )
            print(
                f"New sponsor organisations to insert: "
                f"{len(sponsor_dim_to_insert):,}"
            )

            # Insert only genuinely new sponsors.
            insert_dataframe(
                "DimSponsorOrganisation",
                sponsor_dim_to_insert
            )

            # Refresh sponsor keys after new sponsor rows are inserted.
            with engine.connect() as conn:
                sponsor_key_df = pd.read_sql(
                    "SELECT SponsorKey, MatchedSponsor "
                    "FROM dbo.DimSponsorOrganisation;",
                    conn
                )

            sponsor_key_map = {
                str(name).strip().casefold(): key
                for name, key in zip(
                    sponsor_key_df["MatchedSponsor"],
                    sponsor_key_df["SponsorKey"]
                )
                if pd.notna(name)
            }

            cf_gold_combined = prepare_gold_for_target(
                cf_gold_sql,
                "CF",
                sponsor_key_map
            )
            gtr_gold_combined = prepare_gold_for_target(
                gtr_gold_sql,
                "GTR",
                sponsor_key_map
            )

            cf_comparison_sql = prepare_comparison_for_target(
                cf_comparison_raw,
                "CF"
            )
            gtr_comparison_sql = prepare_comparison_for_target(
                gtr_comparison_raw,
                "GTR"
            )
            comparison_combined = pd.concat(
                [cf_comparison_sql, gtr_comparison_sql],
                ignore_index=True
            )

            # SQL CHECK-constraint safe RecommendedMethod values.
            if (
                not comparison_combined.empty
                and "RecommendedMethod" in comparison_combined.columns
            ):
                comparison_combined["RecommendedMethod"] = (
                    comparison_combined["RecommendedMethod"].apply(
                        lambda x: (
                            None if pd.isna(x)
                            else "ML"
                            if str(x).strip().lower() == "ml"
                            else "Fuzzy"
                            if str(x).strip().lower()
                            in [
                                "fuzzy",
                                "exact",
                                "fuzzy+ml",
                                "fuzzy + ml",
                                "both",
                                "agreement",
                                "methodsagree"
                            ]
                            else "ML"
                            if (
                                "ml" in str(x).strip().lower()
                                and "fuzzy" not in str(x).strip().lower()
                            )
                            else "Fuzzy"
                            if (
                                "fuzzy" in str(x).strip().lower()
                                or "exact" in str(x).strip().lower()
                            )
                            else None
                        )
                    )
                )

            if (
                not comparison_combined.empty
                and "RecommendedScorePercent"
                in comparison_combined.columns
            ):
                comparison_combined["RecommendedScorePercent"] = (
                    pd.to_numeric(
                        comparison_combined["RecommendedScorePercent"],
                        errors="coerce"
                    )
                    .fillna(0)
                    .clip(lower=0, upper=100)
                    .round(2)
                )

            # All four daily tables are inserted only after all preparation
            # and validation above has succeeded.
            insert_dataframe("DataLoadAudit", audit_df)
            insert_dataframe(
                "Gold_ContractsFinder",
                cf_gold_combined
            )
            insert_dataframe(
                "Gold_GtRResearch",
                gtr_gold_combined
            )
            insert_dataframe(
                "MatchComparison",
                comparison_combined
            )

            # Validate score completeness for the current day.
            with engine.connect() as conn:
                score_check = pd.read_sql(
                    text("""
                        SELECT
                            SourceSystem,
                            AutoValidationStatus,
                            COUNT(*) AS [TotalRows],
                            SUM(CASE
                                WHEN FuzzyScorePercent IS NULL
                                THEN 1 ELSE 0
                            END) AS [MissingFuzzyScore],
                            SUM(CASE
                                WHEN MLScorePercent IS NULL
                                THEN 1 ELSE 0
                            END) AS [MissingMLScore],
                            SUM(CASE
                                WHEN RecommendedScorePercent IS NULL
                                THEN 1 ELSE 0
                            END) AS [MissingRecommendedScore]
                        FROM dbo.MatchComparison
                        WHERE InsertedDate >= :day_start
                          AND InsertedDate < :day_end
                        GROUP BY
                            SourceSystem,
                            AutoValidationStatus
                        ORDER BY
                            SourceSystem,
                            AutoValidationStatus;
                    """),
                    conn,
                    params={
                        "day_start": SQL_DAY_START,
                        "day_end": SQL_DAY_END
                    }
                )

            print("\nLatest-day MatchComparison score validation")
            print(score_check.to_string(index=False))

            if not score_check.empty:
                missing_total = int(
                    pd.to_numeric(
                        score_check["MissingFuzzyScore"],
                        errors="coerce"
                    ).fillna(0).sum()
                    + pd.to_numeric(
                        score_check["MissingMLScore"],
                        errors="coerce"
                    ).fillna(0).sum()
                    + pd.to_numeric(
                        score_check["MissingRecommendedScore"],
                        errors="coerce"
                    ).fillna(0).sum()
                )

                if missing_total != 0:
                    raise RuntimeError(
                        "Current-day SQL score validation found "
                        f"{missing_total} missing score value(s)."
                    )

            SQL_UPLOAD_COMPLETED = True
            print_database_storage_status(
                "SQL storage after successful upload"
            )

    except Exception as error:
        SQL_UPLOAD_ERROR = str(error)

        # If SQL failed after partial autocommitted writes, remove current-day
        # partial rows again. Never touch previous dates.
        try:
            if "daily_timestamp_columns" in locals():
                cleanup_current_day_after_failed_upload(
                    daily_timestamp_columns
                )
        except Exception as recovery_error:
            print(
                "SAFE RECOVERY WARNING:",
                str(recovery_error).splitlines()[0]
            )

        if is_storage_capacity_error(error):
            print("\nSQL CAPACITY LIMIT REACHED.")
            print(
                "The extraction and matching outputs were saved "
                "successfully to Google Drive, but SQL Server has no room "
                "for the complete current-day snapshot."
            )
            print(
                "Current-day partial SQL rows were removed where possible; "
                "previous dates were not deleted."
            )
        else:
            print("\nSQL WRITE DID NOT COMPLETE SAFELY.")
            print("Reason:", str(error).splitlines()[0])
            print(
                "Current-day partial rows were removed where possible; "
                "previous dates were preserved."
            )

        log_runtime_step(
            "SQL upload total",
            sql_total_start,
            "FAILED_SAFE",
            None,
            str(error).splitlines()[0]
        )

        if not SQL_FAIL_SAFE_NO_TRACEBACK:
            raise

    # --------------------------------------------------------------
    # FINAL READ-ONLY DATABASE CHECK
    # This section itself is protected so a reporting/view issue
    # cannot create a red traceback at the end of the notebook.
    # --------------------------------------------------------------
    try:
        print("\nSQL history check by insertion date")
        with engine.connect() as conn:
            history_tables = [
                "Gold_ContractsFinder",
                "Gold_GtRResearch",
                "MatchComparison",
                "DataLoadAudit",
                "DimSponsorOrganisation"
            ]

            for table_name in history_tables:
                try:
                    timestamp_column = (
                        get_daily_timestamp_column(table_name)
                        if table_name != "DimSponsorOrganisation"
                        else "InsertedDate"
                    )

                    history_df = pd.read_sql(
                        f"""
                        SELECT
                            CAST([{timestamp_column}] AS DATE)
                                AS InsertionDate,
                            COUNT(*) AS [RowCount]
                        FROM dbo.[{table_name}]
                        GROUP BY
                            CAST([{timestamp_column}] AS DATE)
                        ORDER BY
                            InsertionDate DESC;
                        """,
                        conn
                    )

                    print(f"\ndbo.{table_name}")
                    print(
                        "  no rows"
                        if history_df.empty
                        else history_df.to_string(index=False)
                    )
                except Exception as history_error:
                    print(
                        f"dbo.{table_name}: history check skipped - "
                        f"{str(history_error).splitlines()[0]}"
                    )
    except Exception as final_check_error:
        print(
            "Final SQL history check skipped:",
            str(final_check_error).splitlines()[0]
        )

    if SQL_UPLOAD_COMPLETED:
        log_runtime_step(
            "SQL upload total",
            sql_total_start,
            "SUCCESS",
            None,
            ""
        )
        print("\nSQL UPLOAD COMPLETED SUCCESSFULLY.")
    elif SQL_UPLOAD_SAFE_STOP:
        log_runtime_step(
            "SQL upload total",
            sql_total_start,
            "SKIPPED_CAPACITY",
            None,
            SQL_UPLOAD_ERROR or ""
        )
        print(
            "\nPIPELINE COMPLETED, SQL WRITE SAFELY SKIPPED "
            "BECAUSE OF DATABASE CAPACITY."
        )
    else:
        print(
            "\nPIPELINE COMPLETED WITH SQL FAIL-SAFE RECOVERY. "
            "CSV evidence is available in Google Drive."
        )

    save_runtime_log()
    try:
        engine.dispose()
    except Exception:
        pass
else:
    print("SQL upload skipped because RUN_SQL_UPLOAD is False.")
    save_runtime_log()


SQL daily-history mode: previous UTC dates will be preserved. Only rows from 2026-08-10 00:00:00 up to (not including) 2026-08-11 00:00:00 will be replaced if this notebook is rerun on the same UTC date.

SQL preflight passed.
Prepared sponsor organisations: 130
Prepared CF comparison organisations: 222
Prepared GTR comparison organisations: 141

Daily-history timestamp columns:
  dbo.MatchComparison: InsertedDate
  dbo.Gold_ContractsFinder: InsertedDate
  dbo.Gold_GtRResearch: InsertedDate
  dbo.DataLoadAudit: InsertedDate
SQL storage before same-day replacement: allocated=15.00 MB | used=9.94 MB | free-inside-file=5.06 MB | can-grow=YES

Existing current-day SQL rows before replacement:
  MatchComparison: 363
  Gold_ContractsFinder: 80
  Gold_GtRResearch: 175
  DataLoadAudit: 10
dbo.MatchComparison: replaced-day cleanup removed 363 row(s) from 2026-08-10 only
dbo.Gold_ContractsFinder: replaced-day cleanup removed 80 row(s) from 2026-08-10 only
dbo.Gold_GtRResearch: replaced-day clean